# 04. Production MCP 심화 실습: 재시도·재개·캐시·추적을 어떻게 견디는가

03에서는 MCP 서버와 클라이언트가 stdio로 무엇을 주고받는지 보았습니다. 이 노트북은 그 서버가 **운영 환경에서 마주치는 일**을 직접 일으켜 봅니다. 서버 프로세스가 죽었다가 다시 뜨고, 같은 요청이 두 번 오고, 권한 거부와 일시 장애가 나고, 여러 클라이언트가 HTTP로 붙고, 한 요청이 어느 에이전트 실행에서 왔는지 추적해야 하는 상황입니다. 서버는 과정 전체에서 쓰는 요구사항 인터뷰 MCP 서버(`cap_lab/server/server.py`, Tool 7개, 세션 상태 `draft` -> `ready` -> `frozen`) 그대로이고, 견디는지 여부는 말이 아니라 서버가 남긴 근거(Evidence) 파일과 호출 기록으로 확인합니다.

마지막 부분에서는 H1 서버 게이트를 실제 Jev로 돌립니다. 고객이 준 완료 기준이 실행 결과로 판정 가능한지를 서버가 Jev에 묻고, 판정 불가로 나오면 모델을 거치지 않고 **사용자에게 직접** 묻는(MRTR) 구조입니다. 판정기를 쓸 수 없으면 아무것도 기록하지 않는(fail-closed) 동작도 봅니다.

## 수업 시간표

| 구간 | 내용 | 분 |
|---|---|---|
| 0 | 준비: 작업 폴더, import, 작은 도우미 | 2 |
| 1 | 영속화와 재생: 서버를 다시 띄워도 세션이 이어지는가 | 7 |
| 2 | 멱등성과 중복 호출, R3 중복 규칙 | 5 |
| 3 | 장애 주입과 오류 코드 기반 재시도 | 6 |
| 4 | Streamable HTTP로 띄우고 끄기 | 5 |
| 5 | ttlMs와 cacheScope | 4 |
| 6 | 요청 `_meta`의 traceparent | 5 |
| 7 | H1 게이트: fail-closed와 MRTR (유료 Jev 2회) | 7 |
| 8 | 인증은 개념만 (CIMD) | 2 |
| 9 | 에이전트 1회 실행 (유료) | 3 |
| 10 | 직접 해보기, 확인 질문, 정리 | 4 |

## 이 노트북이 확인하는 것

| 절 | 질문 | 무엇으로 확인하는가 |
|---|---|---|
| 1 | 서버 프로세스가 바뀌어도 같은 세션을 이어 가는가 | `get_session` 결과, 호출 기록의 `server_pid`, SQLite 이벤트와 `replay` |
| 1 | DB가 없어도 근거 파일만으로 복원되는가, 파일을 고치면 어떻게 되는가 | 새 작업 폴더에서 `get_session`, `evidence_conflict`·`evidence_corrupt` |
| 2 | 같은 요청이 두 번 오면 상태가 두 번 바뀌는가 | `event_count`, Tool의 `idempotentHint`, 근거 폴더의 상태 이벤트 수 |
| 3 | 권한 거부와 일시 장애는 어떤 모양으로 돌아오고, 재시도는 누가 정하는가 | `--fault`, 오류 코드와 `retryable`, 코드 규칙으로 쓴 재시도 함수 |
| 4 | stdio와 Streamable HTTP는 무엇이 다른가 | HTTP 서버 하나에 두 클라이언트, `server_pid`, 종료 코드 |
| 5 | 결과를 캐시해도 되는지는 누가 알려 주는가 | `tools/list` 결과의 `ttl_ms`·`cache_scope`, 캐시 힌트를 준 작은 서버 |
| 6 | 서버가 받은 요청을 에이전트의 추적에 어떻게 잇는가 | 요청 `_meta`의 `traceparent`와 노트북 span의 trace-id |
| 7 | 판정은 어디서 하고, 판정이 불가능하면 무엇을 하는가 | `decision_unavailable`, MRTR 질문, `criteria_not_testable` |
| 9 | Deep Agent가 H1 게이트를 만나면 실제로 어떻게 되는가 | dev-05 1회 실행의 호출 순서, 질문, 검사 결과 |

## 호출 규모

이 노트북의 유료 호출은 세 셀뿐입니다.

- 7절 MRTR 수락 셀: Jev 1회. 서버 프로세스가 Jev를 부릅니다(노트북이 아니라 서버가 부르므로 셀 코드에 Jev 호출이 보이지 않습니다).
- 7절 MRTR 거절 셀: Jev 1회. 마찬가지로 서버 프로세스가 부릅니다.
- 9절 에이전트 실행 셀: Claude 약 9회, Jev 최대 3회, 실행 1회.

0~6절, 7절의 스키마 확인 셀과 fail-closed 셀, 8절은 무료입니다. 서버를 이 컴퓨터에서 띄우고 Tool을 직접 부를 뿐 모델이나 판정기를 부르지 않습니다. 유료 셀은 실행 직전에 `announce(...)`(`cap_lab/notebook.py:36`)로 규모를 한 줄 알립니다. 직접 해보기 3은 `RUN_EXERCISE = False`로 잠겨 있고, 켜면 Jev를 최대 2회 부릅니다. 이 노트북은 Langfuse에 연결하지 않습니다. 반복이 3회보다 적은 결과는 모두 **1회 관찰**이라고 부르고, 개선이나 회귀의 근거로 쓰지 않습니다. 모든 코드는 제공되며 점수는 매기지 않습니다. 목표는 각 셀이 무엇을 증명하는지 자기 말로 설명하는 것입니다.

## 0. 준비

이 노트북은 과정 도우미 몇 개와 서버 코드를 직접 가져다 씁니다.

- `connect_server(workdir, ...)`(`cap_lab/notebook.py:76`): `cap_lab/server/server.py`를 **자식 프로세스로 띄워 stdio로 연결한** `mcp.Client`를 줍니다. 세션 DB는 `workdir/sessions.sqlite3`, 근거 파일은 `workdir/evidence/`에 쓰입니다. `async with` 블록이 끝나면 서버 프로세스도 끝납니다. 그래서 블록을 두 번 열면 서버 프로세스가 두 번 뜹니다. 1절은 이 성질을 이용합니다.
- `new_workdir(label)`(`cap_lab/notebook.py:69`): `outputs/notebook/<label>-<날짜-시각>/` 폴더를 새로 만듭니다. 이 노트북은 실험마다 새 폴더를 써서 앞 실험의 상태가 섞이지 않게 합니다.
- `show_events(evidence_dir)`(`cap_lab/notebook.py:99`): 근거 폴더의 이벤트 파일(`event-*.json`)을 표로 보여 줍니다. 시간 칸은 사건이 일어난 시각이 아니라 파일로 내보낸 시각입니다(`cap_lab/server/evidence.py:76`).
- `FaultPlan`(`cap_lab/server/server.py:64`), `replay`(`cap_lab/server/workflow.py:35`)는 서버 코드에서 바로 가져옵니다.

노트북 안에서 정의하는 도우미는 세 개입니다. `body`는 Tool 결과에서 업무 결과(`structuredContent`)를 꺼내고, `outcome`은 그것을 `success` 또는 오류 코드 한 단어로 줄이고, `show_server_calls`는 서버가 남긴 **호출 기록**(`call-*.json`)을 시각 순서로 보여 줍니다. 호출 기록은 서버의 `record_tools` 미들웨어(`cap_lab/server/server.py:155`)가 모든 `tools/call`마다 남기는 감사 기록이고, 어느 서버 프로세스가 처리했는지(`server_pid`, `cap_lab/server/evidence.py:274`)도 담습니다.

이 노트북의 셀은 `await`와 `async with`를 셀 맨 바깥에서 바로 씁니다. Jupyter 커널은 이미 이벤트 루프 안에서 돌기 때문에 `asyncio.run(...)`으로 감쌀 필요가 없고, 감싸면 오히려 오류가 납니다.

In [ ]:
# [따라하기] 준비: 작업 폴더 확인과 import
import asyncio
import json
import os
import shutil
import socket
import sqlite3
import subprocess
import sys
import time
from contextlib import closing
from pathlib import Path

# 1) 커널의 작업 폴더가 Day04_new인지 확인한다. 상대 경로와 .env가 모두 이 폴더 기준이다.
if Path.cwd().name != "Day04_new":
    raise RuntimeError(f"작업 폴더가 Day04_new가 아닙니다: {Path.cwd()}. Day04_new 폴더에서 Jupyter를 다시 여세요.")

# 2) MCP 클라이언트와 사용자 응답 형식(ElicitResult)을 가져온다.
from mcp import Client
from mcp.client.stdio import StdioServerParameters
from mcp_types import ElicitResult

# 3) 과정 도우미와 서버 코드를 가져온다.
from cap_lab.agent import SERVER_SCRIPT, server_argv
from cap_lab.notebook import announce, connect_server, new_workdir, show_events, show_table
from cap_lab.server.server import FaultPlan
from cap_lab.server.workflow import replay
from cap_lab.settings import ROOT

print("작업 폴더:", Path.cwd())
print("서버 스크립트:", SERVER_SCRIPT.relative_to(ROOT))
print("준비 완료")

In [ ]:
# [따라하기] 이 노트북에서만 쓰는 도우미 세 개
def body(result) -> dict:
    """tools/call 결과에서 업무 결과(structuredContent)만 꺼낸다. 없으면 빈 dict를 돌려준다."""
    return result.structured_content or {}


def outcome(content: dict) -> str:
    """업무 결과를 한 단어로 줄인다. 성공이면 'success', 업무 오류면 그 오류 코드."""
    if content.get("success") is True:
        return "success"
    return str(content.get("error", "알 수 없음"))


def show_server_calls(evidence_root: Path) -> list[dict]:
    """서버가 근거 폴더에 남긴 호출 기록(call-*.json)을 기록 시각 순서로 표로 보여 주고, 행 목록을 돌려준다.

    결과 칸은 세 가지 중 하나다.
    - success 또는 업무 오류 코드: Tool 본문이나 장애 주입이 돌려준 결과
    - inputRequests: 서버가 사용자에게 질문하려고 돌려준 중간 결과(7절 MRTR)
    - isError: 입력 형식 검증처럼 업무 코드에 닿기 전에 실패한 결과
    """
    rows = []
    for path in Path(evidence_root).glob("*/call-*.json"):
        record = json.loads(path.read_text(encoding="utf-8"))
        result = record["content"]["result"]
        if isinstance(result, dict) and "inputRequests" in result:
            kind = "inputRequests (사용자에게 질문)"
        elif isinstance(result, dict) and isinstance(result.get("structuredContent"), dict):
            kind = outcome(result["structuredContent"])
        elif isinstance(result, dict) and result.get("isError"):
            kind = "isError"
        else:
            kind = "기타"
        rows.append({
            "기록 시각": record["exported_at"],
            "세션": record["session_id"][:8],
            "도구": record["title"],
            "결과": kind,
            "server_pid": record["content"]["server_pid"],
        })
    rows.sort(key=lambda row: row["기록 시각"])
    if rows:
        show_table(rows)
    else:
        print(f"호출 기록이 없습니다: {evidence_root}")
    return rows


print("도우미 정의 완료: body, outcome, show_server_calls")

## 1. 영속화와 재생: 서버 프로세스가 죽어도 세션이 이어지는가

이 서버는 세션 상태를 **메모리에 두지 않습니다**. 상태를 바꾸는 일은 모두 SQLite `events` 표에 이벤트 한 줄을 덧붙이는 것으로 끝납니다(`cap_lab/server/workflow.py:104`의 표 정의). 이벤트 종류는 세 가지뿐입니다.

| 이벤트 | 언제 생기는가 | payload |
|---|---|---|
| `interview_started` | `start_interview` | `goal` |
| `answer_recorded` | `record_answer` (내용이 바뀔 때만) | `field`, `answer`, 게이트를 켰다면 `gate` |
| `seed_frozen` | `freeze_seed` (처음 확정할 때만) | `seed_id`, `goal`, `constraints`, `acceptance_criteria` |

현재 상태(`draft`/`ready`/`frozen`, 답변, 남은 질문)는 어디에도 저장되지 않습니다. 요청이 올 때마다 `replay`(`cap_lab/server/workflow.py:35`)가 이벤트를 처음부터 순서대로 읽어 **다시 계산**합니다. 그리고 DB에 커밋한 직후 `_publish`(`cap_lab/server/workflow.py:151`)가 같은 이벤트를 근거(Evidence) 파일 `evidence/<세션 ID>/event-0000000N.json`으로 내보냅니다(`cap_lab/server/evidence.py:210`). 파일은 임시 파일에 다 쓴 뒤 링크로 공개하므로 반쯤 쓰인 파일이 보이지 않고, 같은 ID의 기존 파일은 덮어쓰지 않습니다(`cap_lab/server/evidence.py:58`의 `put`).

즉 세션은 **DB와 근거 파일 두 곳**에 남습니다. 요청을 처리할 때 `_load`(`cap_lab/server/workflow.py:134`)가 두 쪽을 맞춰 봅니다. 앞부분이 서로 같아야 하고, 파일 쪽이 더 길면(예: DB가 새것이면) 파일의 나머지를 DB로 가져옵니다.

첫 실험은 단순합니다. 서버 프로세스 1에서 세션을 시작하고 제약 하나를 기록한 뒤 프로세스를 끝냅니다. 그다음 **같은 DB**로 서버 프로세스 2를 띄워 `get_session`을 부릅니다.

**먼저 예측하세요.** 서버 프로세스 2는 프로세스 1의 메모리를 전혀 모릅니다. `get_session`이 돌려줄 `status`와 `event_count`는 무엇일까요? 호출 기록의 `server_pid`는 몇 개의 값으로 나뉠까요?

In [ ]:
# [따라하기] 서버 프로세스 1: 세션을 시작하고 제약을 기록한 뒤 프로세스를 끝낸다
persist_dir = new_workdir("nb04-persist")
GOAL = "출장 영수증을 모아 월별 정산표를 만드는 도구"
CONSTRAINT = "영수증 원본 이미지는 외부로 보내지 않는다"

# 1) async with 블록 안에서만 서버 자식 프로세스가 살아 있다. 블록을 나가면 프로세스가 끝난다.
async with connect_server(persist_dir) as client:
    started = body(await client.call_tool("start_interview", {"goal": GOAL}))
    session_id = started["session_id"]
    recorded = body(await client.call_tool(
        "record_answer", {"session_id": session_id, "field": "constraints", "answer": CONSTRAINT}
    ))

# 2) 서버가 돌려준 상태를 본다. event_count는 지금까지 이 세션에 쌓인 이벤트 수다.
print("세션 ID 앞 8자리:", session_id[:8])
print("start_interview 뒤:", started["status"], "/ 이벤트 수", started["event_count"])
print("record_answer 뒤:", recorded["status"], "/ 이벤트 수", recorded["event_count"])
print("다음 질문:", recorded["next_question"])

# 3) 프로세스가 끝난 뒤에도 남아 있는 것: DB 파일과 근거 폴더.
print("DB 파일 있음:", (persist_dir / "sessions.sqlite3").exists())
print("근거 폴더:", Path(recorded["evidence_dir"]).relative_to(ROOT))

In [ ]:
# [따라하기] 서버 프로세스 2: 같은 DB로 다시 띄워 같은 세션을 이어 간다
# 1) 새 서버 프로세스를 띄워 같은 세션 ID로 조회한다.
async with connect_server(persist_dir) as client:
    resumed = body(await client.call_tool("get_session", {"session_id": session_id}))

print("다시 띄운 서버가 돌려준 상태:", resumed["status"], "/ 이벤트 수", resumed["event_count"])
print("기록된 답변:", resumed["answers"])
print("남은 항목:", resumed["missing_fields"])

# 2) 서버가 남긴 호출 기록. server_pid 칸이 둘로 나뉘면 서로 다른 프로세스가 처리했다는 뜻이다.
persist_calls = show_server_calls(persist_dir / "evidence")
print("호출을 처리한 서버 프로세스 수:", len({row["server_pid"] for row in persist_calls}))

# 3) 근거(Evidence) 이벤트 파일. 파일 하나가 이벤트 하나다.
show_events(persist_dir / "evidence")

두 번째 프로세스가 같은 상태를 돌려준 이유는 상태가 프로세스 안에 있지 않았기 때문입니다. 서버 프로세스는 언제 죽어도 되는 **계산기**이고, 기록은 DB와 근거 파일에 있습니다. 호출 기록의 `server_pid`가 두 값으로 나뉜 것이 그 증거입니다. `get_session`은 조회처럼 보이지만 읽기 전용이 아니라는 점도 눈여겨보세요. 파일에만 있는 기록을 DB로 복원하고 근거 파일을 다시 내보낼 수 있기 때문에 서버는 이 Tool의 `readOnlyHint`를 false로 두었습니다(`cap_lab/server/server.py:312-316`).

`replay`는 DB도 MCP도 모르는 순수 함수입니다. 아래 셀은 SQLite에서 이벤트를 직접 읽어 노트북 안에서 `replay`를 부릅니다. 서버가 하는 일과 정확히 같은 계산입니다. 이벤트를 앞에서부터 몇 개까지만 넘기면 "그 시점까지만 일어난 세계"의 상태가 나옵니다. 운영에서 이것이 중요한 이유는 장애 조사 때문입니다. 어느 이벤트 뒤에 상태가 이상해졌는지 이벤트를 하나씩 늘려 가며 재생해 볼 수 있습니다.

In [ ]:
# [바꿔보기] DB의 이벤트를 직접 읽어 replay로 상태를 다시 계산한다
# 바꿔볼 곳: UPTO를 1로 바꾸면 첫 이벤트까지만 재생한다. 0으로 바꾸면 무엇이 나올지 먼저 예측해 보세요.
UPTO = None  # None이면 모든 이벤트를 재생한다.

# 1) 서버가 쓰는 것과 같은 SQL로 이벤트를 순서대로 읽는다(cap_lab/server/workflow.py:128의 _events).
with closing(sqlite3.connect(persist_dir / "sessions.sqlite3")) as db:
    stored_events = db.execute(
        "SELECT kind, payload FROM events WHERE session_id = ? ORDER BY sequence", (session_id,)
    ).fetchall()

print("DB에 쌓인 이벤트:")
for number, (kind, payload) in enumerate(stored_events, start=1):
    print(f"  {number}. {kind} {payload}")

# 2) 고른 만큼만 replay에 넘긴다.
selected_events = stored_events if UPTO is None else stored_events[:UPTO]
try:
    replayed = replay(session_id, selected_events)
    print(f"\n이벤트 {len(selected_events)}개를 재생한 상태:", replayed["status"])
    print("답변:", replayed["answers"], "/ 남은 항목:", replayed["missing_fields"])
except Exception as error:
    # 이벤트가 0개면 replay가 session_not_found 업무 오류를 낸다(cap_lab/server/workflow.py:37).
    print("\nreplay 오류:", type(error).__name__, getattr(error, "result", error))

### DB가 사라져도: 근거 파일만으로 복원한다

이번에는 DB 없이 근거 폴더만 새 작업 폴더로 복사합니다. 다른 컴퓨터나 다른 담당자에게 세션을 넘길 때 파일만 건네는 상황입니다. 새 서버가 빈 DB로 뜨고 `get_session`이 오면 `_load`가 파일 쪽 이벤트가 더 많다는 것을 보고, 먼저 `replay`로 파일 기록이 온전한지 검사한 뒤(`cap_lab/server/workflow.py:145`) 나머지 이벤트를 DB로 가져옵니다(`cap_lab/server/workflow.py:146-148`). 손상된 기록은 DB로 들어가지 않습니다.

이 성질은 06 노트북의 평가 사례 dev-08(`resume_from_evidence`)이 재는 것이기도 합니다. 실행기는 선행 사례와 본 사례에 **DB를 따로** 주고 근거 폴더만 같이 쓰게 해서, 에이전트가 DB가 아니라 근거 파일로 이전 세션을 이어 가는지를 봅니다(`cap_lab/experiment.py:157`).

In [ ]:
# [따라하기] 근거 폴더만 새 작업 폴더로 복사하고, 빈 DB로 서버를 띄워 세션을 복원한다
restore_dir = new_workdir("nb04-restore")

# 1) 근거 폴더만 복사한다. DB 파일은 가져오지 않는다.
shutil.copytree(persist_dir / "evidence", restore_dir / "evidence")
print("복사 직후 DB 파일 있음:", (restore_dir / "sessions.sqlite3").exists())

# 2) 빈 DB로 서버를 띄우고 같은 세션 ID로 조회한다.
async with connect_server(restore_dir) as client:
    restored = body(await client.call_tool("get_session", {"session_id": session_id}))

print("복원한 상태:", restored["status"], "/ 이벤트 수", restored["event_count"])
print("복원한 답변:", restored["answers"])

# 3) 새 DB에 이벤트가 실제로 들어왔는지 센다.
with closing(sqlite3.connect(restore_dir / "sessions.sqlite3")) as db:
    imported_count = db.execute("SELECT COUNT(*) FROM events WHERE session_id = ?", (session_id,)).fetchone()[0]
print("새 DB로 가져온 이벤트 수:", imported_count)

### 근거 파일을 손으로 고치면

근거 파일은 사람이 읽을 수 있는 JSON입니다. 그렇다면 누군가 파일을 열어 답변을 고치거나, 실수로 파일 하나를 지우면 어떻게 될까요? 아래 셀은 두 가지를 일부러 망가뜨립니다.

- (가) DB가 있는 `restore_dir`에서 두 번째 이벤트 파일의 답변 문장을 바꾼다.
- (나) DB가 없는 새 폴더에 원래 근거를 복사하고, 첫 이벤트 파일(인터뷰 시작 기록)을 지운다.

**먼저 예측하세요.** (가)에서 서버는 DB와 파일 중 어느 쪽을 믿을까요, 아니면 둘 다 믿지 않을까요? (나)에서는 남은 파일만으로 복원할까요?

In [ ]:
# [깨뜨리기] 근거 파일을 고치거나 지우고 get_session을 부른다
# (가) DB가 있는 restore_dir에서 두 번째 이벤트 파일의 답변을 바꾼다.
tampered_file = restore_dir / "evidence" / session_id / "event-00000002.json"
tampered_record = json.loads(tampered_file.read_text(encoding="utf-8"))
tampered_record["content"]["payload"]["answer"] = "몰래 바꾼 제약"
tampered_file.write_text(json.dumps(tampered_record, ensure_ascii=False, indent=2), encoding="utf-8")

async with connect_server(restore_dir) as client:
    conflict = body(await client.call_tool("get_session", {"session_id": session_id}))
print("(가) 파일의 답변만 고쳤을 때:", outcome(conflict), "/", conflict.get("message"))

# (나) DB 없는 새 폴더에 원래 근거를 복사하고, 첫 이벤트 파일을 지운다.
gap_dir = new_workdir("nb04-gap")
shutil.copytree(persist_dir / "evidence", gap_dir / "evidence")
(gap_dir / "evidence" / session_id / "event-00000001.json").unlink()

async with connect_server(gap_dir) as client:
    gap = body(await client.call_tool("get_session", {"session_id": session_id}))
print("(나) 첫 이벤트 파일을 지웠을 때:", outcome(gap), "/", gap.get("message"))

# DB 쪽은 그대로인지 확인한다. 서버는 어느 쪽도 고쳐 쓰지 않는다.
with closing(sqlite3.connect(restore_dir / "sessions.sqlite3")) as db:
    db_answer = db.execute(
        "SELECT payload FROM events WHERE session_id = ? AND kind = 'answer_recorded'", (session_id,)
    ).fetchone()[0]
print("(가) 뒤 DB에 남은 답변 payload:", db_answer)

**관찰할 것.** (가)는 `evidence_conflict`, (나)는 `evidence_corrupt`로 끝났을 것입니다. 두 경우 모두 서버는 **어느 쪽도 고르지 않고 멈춥니다**.

- (가) `_load`는 DB 이벤트와 파일 이벤트를 앞에서부터 하나씩 비교하고, 하나라도 다르면 `evidence_conflict`를 냅니다(`cap_lab/server/workflow.py:139-143`). "파일이 더 최신이겠지"라고 DB를 덮어쓰거나 "DB가 원본이겠지"라고 파일을 덮어쓰면, 누가 무엇을 바꿨는지 알 수 있는 유일한 기록이 사라집니다. DB의 답변 payload가 원래 문장 그대로인 것도 확인하세요.
- (나) `load_events`는 파일 이름의 번호가 1부터 빈틈없이 이어지는지와 형식을 검사합니다(`cap_lab/server/evidence.py:240-253`). 첫 파일이 빠지면 남은 `event-00000002`가 1번 자리에 오므로 순서 검사에서 걸립니다. 복원하지 않았으니 DB에도 아무것도 들어가지 않았습니다.

서버가 스스로 복구하는 경로는 따로 있습니다. DB 커밋 뒤 파일 내보내기만 실패한 경우 `evidence_publish_failed`와 `state_saved: true`를 돌려주고 `get_session`으로 다시 내보내라고 알려 줍니다(`cap_lab/server/workflow.py:156-164`). 설계된 장애는 복구하고, 설계 밖의 변조는 거절합니다. 운영에서 "조용히 고쳐 주는" 서버보다 "멈추고 이유를 말하는" 서버가 나은 이유가 여기 있습니다.

## 2. 멱등성과 중복 호출

운영에서는 같은 요청이 두 번 옵니다. 클라이언트가 응답을 기다리다 시간 초과로 다시 보내고, 에이전트가 결과를 놓쳤다고 생각해 같은 Tool을 또 부르고, 7절에서 볼 MRTR은 같은 `tools/call`을 **설계상** 한 번 더 보냅니다. 같은 요청을 두 번 보내도 한 번 보낸 것과 상태가 같으면 그 호출은 **멱등(idempotent)** 하다고 합니다.

이 서버가 각 Tool에 붙인 `idempotentHint`와 그 근거 줄은 다음과 같습니다.

| Tool | idempotentHint | 왜 그런가 | 근거 줄 |
|---|---|---|---|
| `start_interview` | false | 호출마다 새 세션 ID를 만든다 | `cap_lab/server/server.py:248-253` |
| `record_answer` | true | 같은 답의 재전송은 이벤트를 늘리지 않는다 | `cap_lab/server/server.py:302-310`, `cap_lab/server/workflow.py:204-209` |
| `get_session` | true | 빠진 기록만 복원하고 기존 파일을 덮어쓰지 않는다 | `cap_lab/server/server.py:312-320` |
| `freeze_seed` | true | 이미 확정한 세션은 같은 명세를 돌려준다 | `cap_lab/server/server.py:322-328` |
| `save_evidence` | true | 제목·본문·원본 ID로 산출물 ID를 만들어 같은 내용은 같은 파일이 된다 | `cap_lab/server/server.py:340-351`, `cap_lab/server/evidence.py:203-205` |
| `list_evidence`, `read_evidence` | (읽기 전용) | `readOnlyHint`만 준다 | `cap_lab/server/server.py:52`, `330-338` |

ToolAnnotations는 클라이언트에게 주는 **힌트일 뿐 강제가 아닙니다**. 실제로 중복을 막는 것은 `workflow.py`의 코드입니다. `record_answer`는 이미 기록된 답과 앞뒤 공백을 뺀 새 답이 같으면 이벤트를 덧붙이지 않습니다(`cap_lab/server/workflow.py:204-209`).

평가 쪽에도 이 규칙에 대응하는 검사가 있습니다. R3 중복 규칙 `side_effect_stable`(`cap_lab/scorers.py:270`)은 실행의 근거 폴더 **전체**에서 상태 이벤트(`interview_started`, `answer_recorded`, `seed_frozen`)를 세어 사례가 기대한 `event_count`와 비교합니다. 예를 들어 dev-07은 시작 1 + 답변 2 + 확정 1 = 4개를 기대합니다. 에이전트가 무엇을 몇 번 부르든 **상태가 몇 번 바뀌었는지**만 봅니다.

**먼저 예측하세요.** 같은 제약을 두 번 보내면 `event_count`는 몇이 될까요? 앞뒤 공백만 다른 제약은? 내용이 바뀐 제약은?

In [ ]:
# [따라하기] 같은 답, 공백만 다른 답, 바뀐 답을 차례로 보내고 이벤트 수를 본다
idem_dir = new_workdir("nb04-idempotent")
attempts = [
    ("처음 보내는 제약", "개인정보는 저장하지 않는다"),
    ("같은 제약 재전송", "개인정보는 저장하지 않는다"),
    ("앞뒤 공백만 다른 제약", "  개인정보는 저장하지 않는다  "),
    ("내용이 바뀐 제약", "개인정보와 결제 정보는 저장하지 않는다"),
]

idem_rows = []
async with connect_server(idem_dir) as client:
    # 1) 세션을 하나 시작한다.
    idem_session = body(await client.call_tool("start_interview", {"goal": "고객 문의를 유형별로 나누는 도구"}))["session_id"]
    # 2) 같은 field에 네 가지 답을 차례로 보낸다.
    for label, answer in attempts:
        content = body(await client.call_tool(
            "record_answer", {"session_id": idem_session, "field": "constraints", "answer": answer}
        ))
        idem_rows.append({
            "보낸 것": label,
            "결과": outcome(content),
            "event_count": content.get("event_count"),
            "기록된 제약": content.get("answers", {}).get("constraints"),
        })
    # 3) 같은 서버의 Tool 목록에서 멱등 힌트를 읽는다.
    stdio_listing = await client.list_tools()

show_table(idem_rows)
show_table([
    {
        "Tool": tool.name,
        "idempotentHint": tool.annotations.idempotent_hint if tool.annotations else None,
        "readOnlyHint": tool.annotations.read_only_hint if tool.annotations else None,
    }
    for tool in stdio_listing.tools
])
show_events(idem_dir / "evidence")

In [ ]:
# [깨뜨리기] 응답을 못 받았다고 가정하고 start_interview를 한 번 더 보낸다
dup_dir = new_workdir("nb04-duplicate-start")

async with connect_server(dup_dir) as client:
    first_start = body(await client.call_tool("start_interview", {"goal": GOAL}))
    second_start = body(await client.call_tool("start_interview", {"goal": GOAL}))  # 같은 요청을 다시 보냄
    listed_sessions = body(await client.call_tool("list_evidence", {}))["sessions"]

print("첫 호출이 만든 세션:", first_start["session_id"][:8])
print("재전송이 만든 세션:", second_start["session_id"][:8])
print("근거 폴더의 세션 수:", len(listed_sessions))

# R3 규칙(side_effect_stable)처럼 근거 폴더 전체의 상태 이벤트 파일을 센다.
state_event_files = list((dup_dir / "evidence").glob("*/event-*.json"))
print("상태 이벤트 수:", len(state_event_files), "(인터뷰를 한 번 시작하려던 의도라면 1이어야 한다)")

**관찰할 것.** 같은 목표로 `start_interview`를 두 번 보내자 세션이 두 개 생겼고, 상태 이벤트도 두 개가 되었습니다. 인터뷰를 한 번 시작하려던 실행이었다면 R3 검사(`side_effect_stable`)는 기대보다 이벤트가 많다고 실패로 셉니다. 반대로 위의 `record_answer` 표에서는 같은 답과 공백만 다른 답이 `event_count`를 늘리지 않았고, 내용이 바뀐 답만 이벤트를 하나 더했습니다. 바뀐 답도 이전 답을 지우지 않고 **새 이벤트로 덧붙인다**는 점(근거 이벤트 표의 세 번째 줄)을 확인하세요. 수정 이력이 남습니다.

`start_interview`가 `idempotentHint: false`를 단 것은 "응답을 놓쳤다고 다시 보내면 업무가 중복된다"는 경고입니다. 이 서버는 그 경고만 하고 막지는 않습니다. 막으려면 두 가지 설계가 있습니다. 하나는 클라이언트가 요청마다 고유 키를 보내고 서버가 같은 키에는 같은 세션을 돌려주는 방식(멱등성 키)이고, 다른 하나는 다시 시작하기 전에 `list_evidence`로 목표가 같은 세션이 이미 있는지 먼저 보는 방식(`cap_lab/server/evidence.py:154-171`이 세션 목록과 목표를 함께 돌려줌)입니다. 이 서버는 어느 쪽도 구현하지 않았습니다. 그래서 중복 방지가 에이전트의 판단에 맡겨져 있고, 그 판단을 재는 것이 R3 검사입니다.

In [ ]:
# [바꿔보기] 확정과 산출물 저장도 두 번씩 보내 본다
# 바꿔볼 곳: ARTIFACT_CONTENTS의 두 번째 값을 첫 번째와 다르게 바꾸면 산출물 ID와 파일 수가 어떻게 되는지 보세요.
ARTIFACT_CONTENTS = [
    "1. 문의 유형 목록을 정한다.\n2. 샘플 50건으로 분류 결과를 비교한다.",
    "1. 문의 유형 목록을 정한다.\n2. 샘플 50건으로 분류 결과를 비교한다.",
]

async with connect_server(idem_dir) as client:
    # 1) 남은 질문(완료 기준)에 답하고 두 번 확정한다.
    await client.call_tool("record_answer", {
        "session_id": idem_session, "field": "acceptance_criteria",
        "answer": "샘플 문의 50건 중 45건 이상이 담당 분류와 일치한다",
    })
    first_freeze = body(await client.call_tool("freeze_seed", {"session_id": idem_session}))
    second_freeze = body(await client.call_tool("freeze_seed", {"session_id": idem_session}))
    # 2) 확정 뒤에 답을 바꾸려고 해 본다.
    after_freeze = body(await client.call_tool(
        "record_answer", {"session_id": idem_session, "field": "constraints", "answer": "제약을 바꾸고 싶다"}
    ))
    # 3) 같은 제목으로 산출물을 두 번 저장한다.
    saved_ids = []
    for content in ARTIFACT_CONTENTS:
        saved = body(await client.call_tool("save_evidence", {
            "session_id": idem_session, "title": "구현 계획", "content": content,
            "source_evidence_id": first_freeze["seed_evidence_id"],
        }))
        saved_ids.append(saved["evidence"]["evidence_id"] if saved.get("success") else outcome(saved))

print("첫 확정:", first_freeze["status"], "/ 이벤트 수", first_freeze["event_count"])
print("두 번째 확정의 seed_id가 같은가:", first_freeze["seed"]["seed_id"] == second_freeze["seed"]["seed_id"],
      "/ 이벤트 수", second_freeze["event_count"])
print("확정 뒤 답 수정:", outcome(after_freeze), "/", after_freeze.get("message"))
print("두 번 저장한 산출물 ID:", [item[:18] for item in saved_ids])
print("산출물 파일 수:", len(list((idem_dir / "evidence" / idem_session).glob("artifact-*.json"))))

## 3. 장애 주입: 권한 거부와 일시 장애

실제 장애를 기다릴 수는 없으므로 서버에 장애를 **주입**합니다. 서버의 `--fault` 옵션은 JSON 하나를 받고, `FaultPlan`(`cap_lab/server/server.py:64-71`)이 그것을 검증합니다.

- `permission_denied`: 이 목록의 Tool은 모든 호출을 거절한다.
- `transient`: `{도구 이름: n}` 이면 그 Tool의 처음 n번 호출을 일시 장애로 돌려주고, 그 뒤에는 정상 동작한다.

장애는 `fault_middleware`(`cap_lab/server/server.py:82-105`)가 **업무 코드에 닿기 전에** 돌려줍니다. 그래서 장애 응답에는 부작용이 없고 판정기도 불리지 않습니다. 미들웨어 순서는 바깥부터 `record_tools`, `fault_middleware`이므로(`cap_lab/server/server.py:176-180`) 장애 응답도 호출 기록에 남습니다. 노트북에서는 `connect_server(fault=FaultPlan(...))`이 이것을 `--fault` 인자로 바꿔 넘깁니다(`cap_lab/agent.py:51`의 `server_argv`).

장애 응답의 모양이 중요합니다. `business_failure`(`cap_lab/server/server.py:74-79`)는 프로토콜 오류(`isError: true`)가 아니라 **업무 오류**로 답합니다. `structuredContent`에 `success: false`, 오류 코드(`error`), 사람이 읽을 메시지(`message`)가 들어 있고, 일시 장애에는 `retryable: true`가 붙습니다. 모델과 클라이언트는 이것을 실패한 전송이 아니라 **읽어야 할 결과**로 받습니다.

**먼저 예측하세요.** `record_answer`에 일시 장애 1회를 넣으면 첫 호출과 두 번째 호출의 결과와 `event_count`는 어떻게 될까요? `save_evidence`가 거부되면 근거 폴더에 산출물 파일이 생길까요?

In [ ]:
# [따라하기] 권한 거부와 일시 장애를 넣은 서버를 직접 부른다
fault_dir = new_workdir("nb04-fault")
fault_plan = FaultPlan(permission_denied=["save_evidence"], transient={"record_answer": 1})
print("서버에 넘기는 --fault:", fault_plan.model_dump_json())

fault_rows = []
async with connect_server(fault_dir, fault=fault_plan) as client:
    fault_session = body(await client.call_tool("start_interview", {"goal": GOAL}))["session_id"]
    constraint_call = {"session_id": fault_session, "field": "constraints", "answer": CONSTRAINT}
    planned_calls = [
        ("record_answer 1번째", "record_answer", constraint_call),
        ("record_answer 2번째 (같은 요청)", "record_answer", constraint_call),
        ("save_evidence", "save_evidence", {"session_id": fault_session, "title": "요약", "content": "제약 1건을 기록했다."}),
    ]
    for label, name, arguments in planned_calls:
        result = await client.call_tool(name, arguments)
        content = body(result)
        fault_rows.append({
            "호출": label,
            "isError": result.is_error,
            "결과": outcome(content),
            "retryable": content.get("retryable"),
            "event_count": content.get("event_count"),
            "message": content.get("message", ""),
        })

show_table(fault_rows)
print("산출물 파일 수:", len(list((fault_dir / "evidence" / fault_session).glob("artifact-*.json"))))
fault_calls = show_server_calls(fault_dir / "evidence")

### 재시도는 오류 코드로 정하는 코드 규칙이다

일시 장애를 받은 뒤 같은 요청을 다시 보낼지는 누가 정해야 할까요? 이 서버는 판단에 필요한 모든 것을 **구조화된 값**으로 줍니다. 오류 코드가 `temporarily_unavailable`이고 `retryable`이 `true`면 잠시 뒤 다시 보내고, `permission_denied`면 몇 번을 보내도 결과가 같으니 멈추고 사용자에게 알립니다. 표에 없는 업무 오류(`empty_answer`, `seed_incomplete` 등)는 같은 요청을 다시 보낼 것이 아니라 입력을 고쳐야 합니다.

같은 오류 코드에는 언제나 같은 행동이 맞습니다. 애매함이 없으니 판단 모델이 필요 없습니다. 그래서 사전 등록 P2는 재시도 라우팅을 "Jev 판단이 아니라 코드 규칙"인 **교육용 반례**로 남겼습니다(`docs/DESIGN.md`의 PREREG 블록 P2 마지막 줄). 7절의 H1 게이트와 비교해 보세요. "이 완료 기준을 실행 결과로 참/거짓 판정할 수 있는가"는 자유로운 문장에 대한 판단이고 그것을 푸는 결정적 규칙이 없습니다. 그쪽이 Jev의 자리입니다.

평가 검사도 이 구분을 따릅니다. `no_blind_retry`(`cap_lab/scorers.py:240`)는 거부된 Tool을 같은 인자로 허용치(dev-09는 `max_same_call: 1`)보다 많이 부르면 실패로 세고, `retry_bounded`(`cap_lab/scorers.py:252`)는 일시 장애 Tool의 호출이 `실패 횟수 + 2`를 넘지 않고 결국 답변이 이벤트에 기록되었는지를 봅니다. 아래 함수는 이 규칙을 코드로 옮긴 것입니다.

In [ ]:
# [따라하기] 오류 코드 표만 보고 재시도를 정하는 함수
# 오류 코드 -> 행동. 표에 없는 코드는 'stop'(입력을 고쳐야 하는 업무 오류)으로 본다.
RETRY_RULES = {
    "temporarily_unavailable": "retry",  # 서버가 retryable=True로 알려 준 일시 장애
    "permission_denied": "stop",         # 다시 보내도 결과가 같다. 사용자에게 알린다.
}


async def call_with_retry(client, name: str, arguments: dict, *, max_attempts: int = 3) -> tuple[dict, int]:
    """오류 코드 표(RETRY_RULES)와 retryable 표시만 보고 재시도 여부를 정한다. 판단 모델을 부르지 않는다.

    반환: (마지막 업무 결과, 보낸 횟수). 성공하거나 'stop' 코드를 받으면 바로 끝낸다.
    """
    content: dict = {}
    for attempt in range(1, max_attempts + 1):
        content = body(await client.call_tool(name, arguments))
        if content.get("success") is not False:
            return content, attempt
        rule = RETRY_RULES.get(content.get("error"), "stop")
        if rule != "retry" or content.get("retryable") is not True:
            return content, attempt
        await asyncio.sleep(0.2 * attempt)  # 조금씩 더 길게 물러난 뒤 다시 보낸다.
    return content, max_attempts


# 일시 장애 2회 + 권한 거부를 넣은 서버에서 써 본다.
rule_dir = new_workdir("nb04-retry-rule")
rule_rows = []
async with connect_server(rule_dir, fault=FaultPlan(permission_denied=["save_evidence"],
                                                    transient={"record_answer": 2})) as client:
    rule_session = body(await client.call_tool("start_interview", {"goal": GOAL}))["session_id"]
    answered, sent = await call_with_retry(client, "record_answer",
                                           {"session_id": rule_session, "field": "constraints", "answer": CONSTRAINT})
    rule_rows.append({"Tool": "record_answer", "마지막 결과": outcome(answered), "보낸 횟수": sent})
    saved, sent = await call_with_retry(client, "save_evidence",
                                        {"session_id": rule_session, "title": "요약", "content": "제약 1건을 기록했다."})
    rule_rows.append({"Tool": "save_evidence", "마지막 결과": outcome(saved), "보낸 횟수": sent})

show_table(rule_rows)

In [ ]:
# [깨뜨리기] 오류 코드를 보지 않고 거부된 요청을 무조건 5번 다시 보낸다
blind_dir = new_workdir("nb04-blind-retry")
async with connect_server(blind_dir, fault=FaultPlan(permission_denied=["save_evidence"])) as client:
    blind_session = body(await client.call_tool("start_interview", {"goal": GOAL}))["session_id"]
    blind_arguments = {"session_id": blind_session, "title": "요약", "content": "같은 요약을 계속 보낸다."}
    for attempt in range(1, 6):
        blind_content = body(await client.call_tool("save_evidence", blind_arguments))
        print(f"{attempt}번째 save_evidence:", outcome(blind_content))

blind_calls = [row for row in show_server_calls(blind_dir / "evidence") if row["도구"] == "save_evidence"]
print("같은 인자로 보낸 save_evidence:", len(blind_calls), "번 / dev-09가 허용하는 같은 호출 수: 1")
print("산출물 파일 수:", len(list((blind_dir / "evidence" / blind_session).glob("artifact-*.json"))))

**관찰할 것.** 무조건 다시 보낸 다섯 번은 모두 `permission_denied`였고 산출물은 하나도 생기지 않았습니다. 서버는 다섯 번의 거부를 모두 호출 기록에 남겼습니다. 에이전트가 이렇게 행동하면 `no_blind_retry`가 실패합니다. 반대로 바로 위의 `call_with_retry`는 같은 장애에서 한 번 보내고 멈췄고, 일시 장애 2회는 세 번째 시도에서 통과했습니다. 차이는 영리함이 아니라 **오류 코드를 읽었느냐**입니다. 운영에서 무조건 재시도는 비용(호출 수)만 늘리는 것이 아니라, 거부된 쓰기를 반복해 감사 기록을 오염시키고 레이트 리밋을 앞당깁니다.

장애 응답이 `isError: false`였던 것도 다시 보세요. 업무 오류를 프로토콜 오류로 보내면 클라이언트에 따라 결과 본문을 모델에 보여 주지 않거나 예외로 바꿉니다. 그러면 모델은 `retryable` 같은 구조화된 단서를 읽을 수 없습니다.

In [ ]:
# [깨뜨리기] 장애 계획에 도구 이름을 잘못 쓴다
# 1) 노트북 쪽: FaultPlan이 모르는 도구 이름을 거절한다.
try:
    FaultPlan(permission_denied=["save_evidense"])  # save_evidence의 오타
    print("오타가 통과했습니다(예상과 다름).")
except Exception as error:
    print("FaultPlan 거절:", type(error).__name__)
    for line in str(error).splitlines()[:3]:
        print("   ", line)

# 2) 서버 쪽: 같은 오타를 --fault로 직접 넘기면 서버가 시작 단계에서 멈춘다(cap_lab/server/server.py:420-425).
typo_dir = new_workdir("nb04-fault-typo")
typo_run = subprocess.run(
    [sys.executable, str(SERVER_SCRIPT), "--db", str(typo_dir / "s.sqlite3"),
     "--evidence-dir", str(typo_dir / "evidence"), "--fault", '{"permision_denied": ["save_evidence"]}'],
    cwd=str(ROOT), capture_output=True, text=True, timeout=60, stdin=subprocess.DEVNULL, check=False,
)
print("서버 종료 코드:", typo_run.returncode)
print("서버가 남긴 마지막 줄:", typo_run.stderr.strip().splitlines()[-1])

**관찰할 것.** 오타는 두 곳에서 모두 **시작하기 전에** 걸렸습니다. `FaultPlan`은 도구 이름을 `ToolName` Literal로 받고 모르는 키를 금지하므로(`cap_lab/server/server.py:54-71`의 `extra="forbid"`), 오타 난 계획으로 서버가 떠서 "장애를 넣었는데 아무 일도 없었다"는 잘못된 실험 결과가 나오는 일을 막습니다. 서버 CLI는 argparse 오류로 종료 코드 2를 돌려줍니다. 측정 실험에서 이런 검증은 특히 중요합니다. 장애 주입이 조용히 빠진 R1·R2 실행은 통과처럼 보이지만 아무것도 재지 않은 것입니다.

## 4. Streamable HTTP: 서버를 따로 띄우고 여러 클라이언트가 붙는다

지금까지는 모두 stdio였습니다. 클라이언트가 서버를 **자기 자식 프로세스로** 띄우고 표준 입출력으로 대화합니다(stdout은 MCP 통신 전용이라 서버 진단 로그는 stderr로 갑니다, `cap_lab/server/server.py:434-436`). `connect_server`도, 9절의 Deep Agent(`cap_lab/agent.py:111`의 `PythonStdioTransport`)도 이 방식입니다. 서버는 클라이언트 하나에 하나씩 뜨고 클라이언트와 함께 죽습니다.

Streamable HTTP는 반대입니다. 서버가 **독립된 프로세스**로 먼저 떠 있고, 클라이언트들이 URL로 붙습니다. 이 서버는 `--transport streamable-http --port <포트>`로 시작하면 `http://127.0.0.1:<포트>/mcp`에서 열립니다(`cap_lab/server/server.py:407-418`, `437-439`). 경로 `/mcp`는 SDK 기본값입니다(`.venv/lib/python3.13/site-packages/mcp/server/mcpserver/server.py:1118`). 주소를 `127.0.0.1`(loopback)로 고정한 이유는 인증이 없는 서버이기 때문입니다(`cap_lab/server/server.py:60-61`). 같은 컴퓨터의 클라이언트만 접속할 수 있습니다. 인증은 8절에서 개념으로 다룹니다.

HTTP 서버는 클라이언트가 떠나도 살아 있으므로 **누군가 명시적으로 꺼야** 합니다. 아래 셀은 빈 포트를 하나 받아 서버를 띄우고, 포트가 열릴 때까지 기다린 뒤, 클라이언트 두 개(A, B)를 동시에 붙이고, 마지막에 `terminate`로 끕니다. 끄는 일은 `try/finally` 안에 있어서 중간에 오류가 나도 서버가 남지 않습니다.

**먼저 예측하세요.** A가 시작한 세션을 B가 `get_session`으로 읽을 수 있을까요? 호출 기록의 `server_pid`는 몇 개의 값일까요? 1절의 stdio 실험과 무엇이 다를까요?

In [ ]:
# [따라하기] HTTP 서버를 띄우고, 두 클라이언트로 붙고, 끈다
def free_loopback_port() -> int:
    """127.0.0.1에서 지금 비어 있는 포트 번호를 하나 받는다(운영체제가 고른다)."""
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as probe:
        probe.bind(("127.0.0.1", 0))
        return probe.getsockname()[1]


async def wait_for_port(port: int, timeout_s: float = 30.0) -> float:
    """서버가 포트를 열 때까지 기다리고 걸린 초를 돌려준다. 시간 안에 못 열면 TimeoutError."""
    started_at = time.monotonic()
    while time.monotonic() - started_at < timeout_s:
        try:
            with socket.create_connection(("127.0.0.1", port), timeout=0.5):
                return time.monotonic() - started_at
        except OSError:
            await asyncio.sleep(0.2)
    raise TimeoutError(f"{timeout_s}초 안에 서버가 포트 {port}를 열지 않았습니다.")


# 1) 서버를 독립 프로세스로 띄운다. stdio 서버와 같은 인자에 전송 방식과 포트만 더한다.
http_dir = new_workdir("nb04-http")
http_port = free_loopback_port()
http_command = [
    sys.executable, str(SERVER_SCRIPT),
    *server_argv(http_dir / "sessions.sqlite3", http_dir / "evidence"),
    "--transport", "streamable-http", "--port", str(http_port),
]
print("전송 인자:", " ".join(http_command[-4:]))
http_server = subprocess.Popen(
    http_command, cwd=str(ROOT), stdin=subprocess.DEVNULL, stdout=subprocess.DEVNULL,
    stderr=subprocess.PIPE, text=True,
)

try:
    # 2) 포트가 열리면 클라이언트 두 개를 같은 URL에 붙인다.
    waited_s = await wait_for_port(http_port)
    http_url = f"http://127.0.0.1:{http_port}/mcp"
    print(f"서버가 {waited_s:.1f}초 만에 열렸습니다: {http_url}")
    async with Client(http_url) as client_a, Client(http_url) as client_b:
        http_tools = sorted(tool.name for tool in (await client_a.list_tools()).tools)
        made_by_a = body(await client_a.call_tool("start_interview", {"goal": "A 클라이언트가 시작한 인터뷰"}))
        read_by_b = body(await client_b.call_tool("get_session", {"session_id": made_by_a["session_id"]}))
        print("협상된 프로토콜 버전:", client_a.protocol_version)
    print("B가 읽은 A의 세션 목표:", read_by_b.get("goal"))
    print("HTTP와 stdio의 Tool 목록이 같은가:",
          http_tools == sorted(tool.name for tool in stdio_listing.tools))
finally:
    # 3) 서버를 끈다. 클라이언트가 떠나도 HTTP 서버는 혼자 살아 있기 때문이다.
    http_server.terminate()
    _, http_log = http_server.communicate(timeout=30)

print("서버 종료 코드:", http_server.returncode, "(POSIX에서 음수는 신호로 끝났다는 뜻, -15는 SIGTERM)")
print("서버 로그 중 시작·종료 줄:")
for line in http_log.splitlines():
    if "Uvicorn running" in line or "Shutting down" in line or "Finished server process" in line:
        print("   ", line)

# 4) HTTP로 온 호출도 stdio와 같은 방식으로 호출 기록에 남는다.
http_calls = show_server_calls(http_dir / "evidence")
print("HTTP 호출을 처리한 서버 프로세스 수:", len({row["server_pid"] for row in http_calls}))

결과를 1절과 나란히 놓으면 차이가 보입니다.

| | stdio | Streamable HTTP |
|---|---|---|
| 서버를 띄우는 쪽 | 클라이언트(자식 프로세스) | 운영자(독립 프로세스) |
| 프로세스 수 | 클라이언트마다 하나 | 하나에 여러 클라이언트 |
| 수명 | 클라이언트와 함께 끝남 | 명시적으로 꺼야 끝남 |
| 주소 | 없음(파이프) | `http://127.0.0.1:<포트>/mcp` |
| 인증 | 프로세스를 띄운 사람 = 사용자 | 필요(이 서버는 loopback으로 대신함) |
| 세션 상태 | DB + 근거 파일 | DB + 근거 파일 |

마지막 줄이 이 노트북의 요점입니다. 상태가 프로세스 밖(DB와 파일)에 있기 때문에 전송 방식을 바꿔도 서버 코드는 한 줄도 바뀌지 않았고, 클라이언트 B는 A가 만든 세션을 그대로 읽었습니다. 2026-07-28 프로토콜은 연결 단위 세션(`Mcp-Session-Id`)을 없애고 요청마다 `_meta`에 프로토콜 버전과 capabilities를 싣는 stateless 방식으로 바뀌었습니다(`docs/DESIGN.md` 5절). 서버가 이어 가야 할 업무 상태는 이 서버처럼 **서버가 발급한 핸들**(여기서는 `session_id`)을 도구 인자로 주고받아 찾습니다. 6절에서 실제 요청의 `_meta`를 열어 봅니다.

## 5. ttlMs와 cacheScope (SEP-2549)

클라이언트는 같은 질문을 자주 반복합니다. 에이전트는 실행마다 `tools/list`를 부르고, 근거 목록도 여러 번 읽습니다. 매번 서버까지 가야 할까요? 2026-07-28 프로토콜은 **목록·읽기 결과**에 신선도 힌트 두 개를 싣게 했습니다.

- `ttlMs`: 이 결과를 몇 밀리초 동안 신선하다고 봐도 되는가. HTTP의 `Cache-Control: max-age`와 같은 뜻이고, 0은 "바로 낡음"(캐시하지 말 것)입니다.
- `cacheScope`: `public`이면 공유 캐시가 다른 사용자에게도 이 결과를 줄 수 있고, `private`이면 같은 인가 맥락 안에서만 다시 쓸 수 있습니다.

설치된 mcp 2.3.0에서 확인할 곳은 다음과 같습니다.

| 무엇 | 어디 |
|---|---|
| 두 필드를 가진 결과의 기반 클래스와 기본값 `ttl_ms=0`, `cache_scope="private"` | `.venv/lib/python3.13/site-packages/mcp_types/_types.py:198-211` |
| 힌트를 싣는 메서드 목록(`CacheableMethod`) | `.venv/lib/python3.13/site-packages/mcp_types/methods.py:411-418` |
| 서버가 메서드별 힌트를 정하는 `CacheHint`와 `cache_hints` | `.venv/lib/python3.13/site-packages/mcp/server/caching.py:23-85` |
| 클라이언트 캐시: 기본 `CacheConfig()`가 서버 힌트를 따른다 | `.venv/lib/python3.13/site-packages/mcp/client/client.py:363-370` |

이 과정의 서버는 `MCPServer`를 만들 때 `cache_hints`를 주지 않습니다(`cap_lab/server/server.py:177-180`). 그러면 SDK 기본값이 그대로 나갑니다. 아래 셀은 이 서버가 실제로 무엇을 보내는지 읽습니다.

**먼저 예측하세요.** 이 서버의 `tools/list` 결과에 `ttl_ms`가 실려 올까요? 실려 온다면 값은? `tools/call` 결과(`CallToolResult`)에도 이 필드가 있을까요?

In [ ]:
# [따라하기] 이 서버의 tools/list 결과에 실린 캐시 힌트를 읽는다
from mcp_types import CallToolResult, ListToolsResult
from mcp_types.methods import CACHEABLE_METHODS

cache_dir = new_workdir("nb04-cache")
async with connect_server(cache_dir) as client:
    tools_result = await client.list_tools()

# 1) 서버가 보낸 값. model_fields_set에 이름이 있으면 서버 응답에 그 필드가 실제로 있었다는 뜻이다.
print("tools/list 결과의 ttl_ms:", tools_result.ttl_ms)
print("tools/list 결과의 cache_scope:", tools_result.cache_scope)
print("서버 응답에 두 필드가 실려 왔는가:", {"ttl_ms", "cache_scope"} <= tools_result.model_fields_set)

# 2) 어떤 메서드의 결과가 힌트를 싣는가. tools/call은 목록에 없다.
print("힌트를 싣는 메서드:", sorted(CACHEABLE_METHODS))
print("ListToolsResult에 ttl_ms 필드가 있는가:", "ttl_ms" in ListToolsResult.model_fields)
print("CallToolResult에 ttl_ms 필드가 있는가:", "ttl_ms" in CallToolResult.model_fields)

In [ ]:
# [바꿔보기] 캐시 힌트를 준 작은 서버로 클라이언트 캐시가 실제로 쓰이는지 본다
from mcp.server import MCPServer
from mcp.server.caching import CacheHint

# 바꿔볼 곳: TTL_MS를 0으로 바꾸면 서버에 도착하는 tools/list 수가 어떻게 달라지는지 보세요.
TTL_MS = 60_000
CACHE_SCOPE = "private"

received_methods: list[str] = []


async def count_requests(ctx, call_next):
    """서버에 실제로 도착한 요청의 메서드 이름을 적는다. 응답은 바꾸지 않는다."""
    received_methods.append(ctx.method)
    return await call_next(ctx)


# 1) Tool 하나짜리 시연용 서버. tools/list 결과에 캐시 힌트를 싣는다.
cache_demo = MCPServer(
    name="cache-demo",
    middleware=[count_requests],
    cache_hints={"tools/list": CacheHint(ttl_ms=TTL_MS, scope=CACHE_SCOPE)},
)


@cache_demo.tool()
def ping() -> str:
    """연결 확인용 Tool."""
    return "pong"


# 2) 같은 프로세스 안에서 연결한다(서버 객체를 그대로 넘김). tools/list를 세 번 부른다.
async with Client(cache_demo) as client:
    first_listing = await client.list_tools()
    second_listing = await client.list_tools()
    bypassed_listing = await client.list_tools(cache_mode="bypass")  # 캐시를 건너뛰고 서버로 간다

print("첫 응답의 힌트:", first_listing.ttl_ms, first_listing.cache_scope)
print("서버에 도착한 요청:", received_methods)
print("클라이언트가 부른 tools/list: 3번 / 서버에 도착한 tools/list:", received_methods.count("tools/list"), "번")

이 과정의 서버는 `ttl_ms=0`, `cache_scope="private"`를 보냈고, 두 필드는 서버 응답에 실제로 실려 왔습니다. 2026-07-28 응답에서 두 필드는 필수이고, `cache_hints`를 주지 않은 서버는 SDK 기본값(바로 낡음, 공유 금지)을 보냅니다(`.venv/lib/python3.13/site-packages/mcp_types/_types.py:198-205`의 설명). 그래서 클라이언트는 `tools/list`를 부를 때마다 서버까지 갑니다. 시연용 서버에서는 `ttl_ms=60000`이 붙자 두 번째 `tools/list`가 서버에 도착하지 않았고, `cache_mode="bypass"`만 서버까지 갔습니다.

`tools/call`은 힌트를 싣는 메서드가 아닙니다. 상태를 바꾸는 호출은 캐시가 대신 답하면 안 되기 때문입니다. 이 서버에 TTL을 준다면 어떨까요? 한 서버 프로세스 안에서 Tool 목록은 바뀌지 않지만, 같은 주소의 서버를 다른 옵션으로 다시 띄우면 바뀝니다. 예를 들어 `--gate h1`은 `record_answer`의 `openWorldHint`를 바꾸고(`cap_lab/server/server.py:305-310`), `--interface plain`은 enum과 설명을 뺍니다. TTL이 길면 클라이언트가 옛 목록을 계속 쓸 수 있습니다. `public`은 사용자마다 보이는 Tool이 다를 수 있는 서버(인증 뒤 권한별 목록)에서 다른 사람의 목록을 내줄 수 있으므로 더 조심해야 합니다. 이 서버가 기본값을 둔 것은 그래서 안전한 선택이고, 비용은 매번의 `tools/list` 왕복입니다.

## 6. 요청 `_meta`의 traceparent: 에이전트 추적과 서버를 잇는 끈

에이전트 실행 하나는 여러 프로세스에 걸칩니다. 노트북(또는 실행기) 프로세스가 Claude를 부르고, MCP 서버 프로세스가 Tool을 실행하고, 서버가 다시 Jev를 부릅니다. Langfuse에서 "이 실행의 이 Tool 호출"을 서버 쪽 기록과 이으려면 요청마다 **같은 추적 ID**가 따라가야 합니다. 그 끈이 W3C Trace Context의 `traceparent`입니다. 모양은 `00-<trace-id 32자리 16진수>-<parent-id 16자리 16진수>-<flags 2자리>`입니다.

mcp 클라이언트는 요청을 보낼 때마다 `MCP send <메서드>`라는 클라이언트 span을 열고, 지금 OpenTelemetry 문맥을 요청의 `_meta`에 주입합니다(`.venv/lib/python3.13/site-packages/mcp/shared/jsonrpc_dispatcher.py:387-393`). 서버 쪽은 `_meta`에서 그것을 꺼내 서버 span의 부모로 삼습니다(`.venv/lib/python3.13/site-packages/mcp/server/_otel.py:35-39`). 주입은 프로세스에 OpenTelemetry `TracerProvider`가 있을 때만 의미가 있습니다. 없으면 span이 기록되지 않고 `_meta`에 아무것도 들어가지 않습니다.

Langfuse 4 SDK는 OpenTelemetry 위에 만들어졌고, 클라이언트를 만들 때 전역 `TracerProvider`가 없으면 하나를 등록합니다(`.venv/lib/python3.13/site-packages/langfuse/_client/resource_manager.py:667-677`). 그래서 Langfuse observation 안에서 에이전트가 MCP Tool을 부르면 `traceparent`의 trace-id가 곧 Langfuse trace id입니다. 환경 점검에서 서버가 받은 trace-id와 Langfuse trace id가 일치하는 것을 확인했습니다(`docs/DESIGN.md:62`). 이 노트북은 Langfuse 대신 **메모리에만 모으는** OpenTelemetry 추적기를 커널에 켜고 같은 원리를 봅니다.

한 가지 함정이 있습니다. 이 서버의 호출 기록은 `_meta`를 **일부러 남기지 않습니다**. `record_tools`는 요청에서 `name`과 `arguments`만 꺼내 기록합니다(`cap_lab/server/server.py:159-161`, "메타데이터·환경변수·통신 헤더는 기록하지 않는다"). 그래서 근거 폴더의 호출 기록에서는 traceparent를 읽을 수 없습니다. 서버가 받은 traceparent를 보려면 서버에 관찰용 미들웨어를 하나 더 붙여야 합니다. `MCPServer.middleware`는 관찰용 `async (ctx, call_next)` 함수를 덧붙이라고 공개된 목록입니다(`.venv/lib/python3.13/site-packages/mcp/server/mcpserver/server.py:278-287`). 아래 셀은 `create_server`로 과정 서버를 그대로 만들고 `_meta`를 파일에 적는 미들웨어 하나만 덧붙이는 **관찰용 실행 파일**을 작업 폴더에 씁니다. 서버 코드는 바꾸지 않습니다.

알려진 빈틈: 서버 프로세스에는 `TracerProvider`도 내보내기(exporter)도 없습니다. SDK가 서버 span을 만들기는 하지만 아무 데도 가지 않으므로 **Langfuse에는 아직 서버 쪽 span이 보이지 않습니다**(`docs/DESIGN.md:71`의 미확인 항목). Langfuse에 남는 것은 에이전트 프로세스 쪽 span뿐입니다.

In [ ]:
# [따라하기] 커널에 OpenTelemetry 추적기를 켜고, 관찰용 서버 실행 파일을 만든다
from opentelemetry import trace
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import SimpleSpanProcessor
from opentelemetry.sdk.trace.export.in_memory_span_exporter import InMemorySpanExporter

# 1) 전역 TracerProvider는 프로세스에 한 번만 정할 수 있다. 이 셀을 다시 실행하면 이미 정한 것을 그대로 쓴다.
kernel_provider = trace.get_tracer_provider()
if isinstance(kernel_provider, trace.ProxyTracerProvider):
    kernel_provider = TracerProvider()
    trace.set_tracer_provider(kernel_provider)

# 2) 끝난 span을 메모리에만 모은다. 아무 데도 보내지 않는다(Langfuse가 아니다).
span_exporter = InMemorySpanExporter()
kernel_provider.add_span_processor(SimpleSpanProcessor(span_exporter))
tracer = trace.get_tracer("day04-nb04")

# 3) 관찰용 서버 실행 파일. 과정 서버를 그대로 만들고 _meta를 적는 미들웨어 하나만 덧붙인다.
OBSERVER_SOURCE = r'''"""관찰용 서버 실행 파일. cap_lab 서버를 그대로 만들고, 요청 _meta를 파일에 적는 미들웨어 하나만 덧붙인다.

인자: <Day04_new 경로> <DB 경로> <근거 폴더> <_meta 기록 파일>
"""
import json
import sys
from pathlib import Path

sys.path.insert(0, sys.argv[1])
from cap_lab.server.server import create_server

db_path, evidence_dir, meta_log = Path(sys.argv[2]), Path(sys.argv[3]), Path(sys.argv[4])
server = create_server(db_path, evidence_dir)


async def write_meta(ctx, call_next):
    """요청마다 메서드 이름과 _meta를 한 줄씩 적는다. 응답은 바꾸지 않는다."""
    meta = dict(ctx.meta) if ctx.meta is not None else {}
    with meta_log.open("a", encoding="utf-8") as log:
        log.write(json.dumps({"method": ctx.method, "meta": meta}, ensure_ascii=False) + "\n")
    return await call_next(ctx)


server.middleware.append(write_meta)
server.run(transport="stdio")
'''

trace_dir = new_workdir("nb04-trace")
observer_script = trace_dir / "observe_server.py"
observer_script.write_text(OBSERVER_SOURCE, encoding="utf-8")
meta_log = trace_dir / "meta.jsonl"
observer_params = StdioServerParameters(
    command=sys.executable,
    args=[str(observer_script), str(ROOT), str(trace_dir / "sessions.sqlite3"), str(trace_dir / "evidence"), str(meta_log)],
    cwd=str(ROOT),
)
print("커널의 TracerProvider:", type(kernel_provider).__name__)
print("관찰용 실행 파일:", observer_script.relative_to(ROOT))

In [ ]:
# [따라하기] 부모 span 아래에서 Tool을 부르고, 서버가 받은 traceparent를 읽는다
def parse_traceparent(value: str) -> dict:
    """traceparent 문자열을 네 칸으로 나눈다. 형식이 아니면 빈 칸으로 돌려준다."""
    parts = value.split("-") if value else []
    if len(parts) != 4:
        return {"version": "", "trace-id": "", "parent-id": "", "flags": ""}
    return dict(zip(("version", "trace-id", "parent-id", "flags"), parts))


# 1) 노트북 셀 전체를 하나의 부모 span으로 감싸고 그 안에서 Tool을 부른다.
with tracer.start_as_current_span("노트북 셀: 인터뷰 시작") as cell_span:
    async with Client(observer_params) as client:
        traced = body(await client.call_tool("start_interview", {"goal": GOAL}))
cell_trace_id = format(cell_span.get_span_context().trace_id, "032x")
print("노트북 셀 span의 trace-id:", cell_trace_id)

# 2) 서버가 받은 _meta. 요청마다 traceparent가 왔는지, trace-id가 셀 span과 같은지 본다.
received_meta = [json.loads(line) for line in meta_log.read_text(encoding="utf-8").splitlines()]
show_table([
    {"메서드": item["method"], **parse_traceparent(item["meta"].get("traceparent", "")),
     "셀 span과 같은 trace": parse_traceparent(item["meta"].get("traceparent", ""))["trace-id"] == cell_trace_id}
    for item in received_meta
])
tools_call_meta = next(item["meta"] for item in received_meta if item["method"] == "tools/call")
print("tools/call 요청 _meta의 키:", sorted(tools_call_meta))

# 3) 클라이언트 쪽 span(MCP send ...). 이 span의 id가 traceparent의 parent-id다.
show_table([
    {"span 이름": span.name, "span id": format(span.context.span_id, "016x"),
     "셀 span과 같은 trace": format(span.context.trace_id, "032x") == cell_trace_id}
    for span in span_exporter.get_finished_spans() if span.name.startswith("MCP send")
])

# 4) 서버의 호출 기록(근거)에는 _meta가 없다. request에는 name과 arguments만 남는다.
traced_call_file = next((trace_dir / "evidence" / traced["session_id"]).glob("call-*.json"))
print("호출 기록 request의 키:", list(json.loads(traced_call_file.read_text(encoding="utf-8"))["content"]["request"]))

세 표를 이어 읽어 보세요. 서버가 받은 모든 요청의 trace-id가 노트북 셀 span의 trace-id와 같고, 각 요청의 parent-id는 그 요청을 보낸 클라이언트 span(`MCP send tools/call start_interview` 등)의 span id와 같습니다. 즉 서버는 "어느 실행의 어느 요청인지"를 받았습니다. `_meta`에는 traceparent 말고도 `io.modelcontextprotocol/protocolVersion`과 클라이언트 정보·capabilities가 들어 있습니다. 4절에서 말한 stateless 방식, 곧 요청마다 버전과 capabilities를 싣는 모습입니다.

그런데 근거 폴더의 호출 기록 `request`에는 `name`과 `arguments`뿐입니다. 이 서버의 근거만으로는 "이 호출 기록이 Langfuse의 어느 trace였는가"를 알 수 없습니다. 이것을 잇는 방법은 두 가지입니다. (1) 서버 프로세스에도 TracerProvider와 exporter를 두어 서버 span을 Langfuse로 보낸다, (2) 호출 기록에 trace-id만 골라 남긴다. 이 과정은 둘 다 하지 않았고, 그래서 지금은 **에이전트 쪽 추적과 서버 쪽 근거가 trace-id로 이어지지 않습니다**. 서버가 `_meta`를 통째로 기록하지 않는 데에는 이유가 있습니다. `_meta`에는 클라이언트가 무엇이든 실을 수 있으므로, 통째로 남기면 의도하지 않은 값이 감사 기록에 들어갈 수 있습니다.

In [ ]:
# [깨뜨리기] 부모 span 없이 Tool을 두 번 부르면 traceparent는 어떻게 되는가
meta_log.unlink(missing_ok=True)  # 앞 셀의 기록을 지우고 새로 받는다.

# 1) 이번에는 노트북 쪽 부모 span을 열지 않는다.
async with Client(observer_params) as client:
    await client.call_tool("list_evidence", {})
    await client.call_tool("list_evidence", {})

# 2) 서버가 받은 traceparent의 trace-id를 모은다.
unparented = [json.loads(line) for line in meta_log.read_text(encoding="utf-8").splitlines()]
unparented_ids = [parse_traceparent(item["meta"].get("traceparent", ""))["trace-id"] for item in unparented]
for item, trace_id in zip(unparented, unparented_ids):
    print(f"{item['method']:<16} trace-id: {trace_id or '(없음)'}")
print("서로 다른 trace-id 수:", len(set(unparented_ids)), "/ 요청 수:", len(unparented_ids))

**관찰할 것.** 부모 span이 없어도 traceparent는 왔습니다. 커널에 TracerProvider가 있으므로 클라이언트가 요청마다 `MCP send ...` span을 열기 때문입니다. 하지만 그 span들은 각자 **새 trace의 뿌리**라서 요청마다 trace-id가 다릅니다. Langfuse라면 한 번의 에이전트 실행이 서로 무관한 trace 여러 개로 흩어져 보이는 상황입니다. 여러 Tool 호출을 한 실행으로 묶어 주는 것은 traceparent 자체가 아니라 **그것을 감싸는 부모 span**(실행기의 Langfuse observation)입니다. TracerProvider조차 없는 새 커널에서는 span이 기록되지 않으므로 `_meta`에 traceparent가 아예 들어가지 않습니다. 이 노트북은 그 경우를 셀로 보여 주지 않습니다. 커널에 한 번 정한 TracerProvider는 되돌릴 수 없기 때문입니다.

## 7. H1 게이트: 판정은 서버가 Jev에, 질문은 사용자에게(MRTR)

사전 등록 P2의 H1은 이렇습니다. 서버는 완료 기준(acceptance criterion)에 대해 Jev의 **Score 질문** testability를 묻고, 점수가 문턱값 이상일 때만 기록합니다. 미만이면 서버가 MRTR로 **사용자에게 직접** 묻습니다. 모델은 그 인자를 보거나 채울 수 없습니다. 사용자가 거절하면 `criteria_not_testable`을 돌려줍니다. Jev가 어떤 이유로든 실패하면 `decision_unavailable`을 돌려주고 아무것도 기록하지 않습니다(fail-closed).

3절의 재시도와 달리 여기는 판단의 자리입니다. "분류가 잘 되면 완료"가 실행 결과로 참/거짓을 가릴 수 있는 기준인지는 자유로운 문장에 대한 판단이고, 그것을 푸는 결정적 규칙이 없습니다. 생성 모델(Claude)에게 맡기면 같은 모델이 기준을 받아 적으면서 동시에 그 기준을 심사하게 됩니다. 그래서 판단은 서버 쪽 Jev가, 대체 문장은 사용자가 냅니다.

코드에서 보는 순서는 다음과 같습니다.

1. 게이트를 켜면 `record_answer`에 숨은 매개변수 두 개가 생깁니다(`cap_lab/server/server.py:261-267`). `assessment: Annotated[Assessment, Resolve(assess)]`와 `revision: Annotated[ElicitationResult[RevisedCriteria], Resolve(ask_if_untestable)]`입니다. `Resolve(...)`로 선언한 매개변수는 Tool 본문이 돌기 전에 **서버가** 채우며, 모델이 보는 입력 형식에 나타나지 않습니다.
2. `assess`(`cap_lab/server/server.py:224-233`)는 field가 `acceptance_criteria`일 때만 판정합니다. 판정기는 첫 판정 때 만듭니다(`cap_lab/server/server.py:212-222`). 그래서 게이트를 켠 서버라도 완료 기준이 오기 전에는 Jev를 부르지 않습니다.
3. 질문은 Score 하나(`TESTABILITY`, 세 단계, `cap_lab/server/gate.py:28-35`)이고, 문턱값 기본 0.5는 라벨로 재지 않은 자리값입니다(사전 등록 P4).
4. `ask_if_untestable`(`cap_lab/server/server.py:235-246`)은 점수가 문턱값 미만일 때만 `Elicit(...)`을 돌려줍니다. 2026-07-28 프로토콜에서 서버는 이것을 `InputRequiredResult`(사용자 질문을 담은 중간 결과)로 돌려주고, `mcp.Client`는 `elicitation_callback`으로 답을 받아 **같은 `tools/call`을 답과 함께 다시 보냅니다**(`.venv/lib/python3.13/site-packages/mcp/client/client.py:769-776`, `.venv/lib/python3.13/site-packages/mcp/server/mcpserver/resolve.py:10-16`). 이 재전송에서 resolver가 다시 돌지만, `TestabilityGate`는 성공한 점수를 문장별로 저장하므로 같은 서버 프로세스에서 같은 문장으로 Jev를 두 번 부르지 않습니다(`cap_lab/server/gate.py:63-70`).

먼저 무료 확인부터 합니다. 게이트를 켜도 모델이 보는 `record_answer`의 입력 인자는 그대로인지 봅니다.

In [ ]:
# [따라하기] 게이트를 켜도 모델이 보는 record_answer 입력 인자는 그대로다 (무료)
gate_schema_dir = new_workdir("nb04-gate-schema")
schema_rows = []
for gate in ("off", "h1"):
    # 게이트 모드마다 서버를 따로 띄운다. 목록만 요청하므로 판정기를 만들지 않는다.
    async with connect_server(gate_schema_dir / gate, gate=gate) as client:
        listed_by_gate = {tool.name: tool for tool in (await client.list_tools()).tools}
    record_tool = listed_by_gate["record_answer"]
    schema_rows.append({
        "gate": gate,
        "모델이 보는 입력 인자": sorted(record_tool.input_schema["properties"]),
        "필수 인자": sorted(record_tool.input_schema.get("required", [])),
        "openWorldHint": record_tool.annotations.open_world_hint if record_tool.annotations else None,
    })
show_table(schema_rows)

### 판정기를 쓸 수 없으면: fail-closed

게이트가 실제로 막는지 보려면 판정이 **실패하는** 경우부터 봐야 합니다. 판정기가 없을 때 기준을 그냥 통과시키는 게이트(fail-open)는 장애가 나는 순간 조용히 꺼진 것과 같습니다. 아래 셀은 가짜 판정기를 쓰지 않고, 서버가 판정기 키를 **찾지 못하게** 만듭니다.

- 서버는 `load_settings`로 키를 읽는데, 이 함수는 `DAY04_ENV_FILE`이 가리키는 파일 하나만 읽습니다(`cap_lab/settings.py:50-56`). 셸에 남은 키는 읽지 않습니다.
- `connect_server`는 자식 프로세스에 `DAY04_ENV_FILE`만 넘기고(`cap_lab/notebook.py:93`), mcp의 stdio 클라이언트는 PATH·HOME 같은 안전한 변수만 물려줍니다(`.venv/lib/python3.13/site-packages/mcp/client/stdio.py:41`, `129`).
- 그래서 작업 폴더에 빈 `.env` 파일을 만들고 `DAY04_ENV_FILE`을 그 파일로 돌리면, 서버는 Decision API 키를 하나도 찾지 못합니다. `make_decider`가 멈추고(`cap_lab/decisions.py:201`), `open_gate`가 그것을 `GateUnavailable`로 바꾸고(`cap_lab/server/server.py:216-220`), Tool 본문이 `decision_unavailable`을 돌려줍니다(`cap_lab/server/server.py:269-274`). 네트워크 호출은 없습니다.

`DAY04_ENV_FILE`은 커널의 환경 변수라서 셀이 끝난 뒤에도 남습니다. 남으면 다음 셀의 MRTR까지 판정기 없이 돌게 되므로, 셀 안의 `try/finally`에서 반드시 원래 값으로 되돌립니다.

**먼저 예측하세요.** 판정기 키가 없는 서버에 모호한 완료 기준을 보내면 (1) 그대로 기록된다, (2) 사용자에게 묻는다, (3) 거절하고 아무것도 기록하지 않는다 중 무엇일까요? 사용자 질문 콜백은 몇 번 불릴까요?

In [ ]:
# [깨뜨리기] 판정기 키가 없는 .env로 서버를 띄운다 (Jev를 부를 수 없다, 무료)
closed_dir = new_workdir("nb04-fail-closed")
empty_env_file = closed_dir / "empty.env"
empty_env_file.write_text("", encoding="utf-8")
closed_questions: list[str] = []


async def note_question_and_decline(context, params):
    """서버가 사용자에게 질문하면 적어 두고 거절한다. fail-closed라면 불리지 않아야 한다."""
    closed_questions.append(params.message)
    return ElicitResult(action="decline")


previous_env_file = os.environ.get("DAY04_ENV_FILE")
os.environ["DAY04_ENV_FILE"] = str(empty_env_file)
try:
    async with connect_server(closed_dir, gate="h1", elicitation_callback=note_question_and_decline) as client:
        closed_session = body(await client.call_tool("start_interview", {"goal": GOAL}))["session_id"]
        await client.call_tool("record_answer", {"session_id": closed_session, "field": "constraints", "answer": "없음"})
        closed_result = body(await client.call_tool("record_answer", {
            "session_id": closed_session, "field": "acceptance_criteria", "answer": "분류가 잘 되면 완료",
        }))
        closed_state = body(await client.call_tool("get_session", {"session_id": closed_session}))
finally:
    # 이 셀 밖에서는 원래 .env를 쓰도록 반드시 되돌린다. 다음 셀은 진짜 Jev를 부른다.
    if previous_env_file is None:
        os.environ.pop("DAY04_ENV_FILE", None)
    else:
        os.environ["DAY04_ENV_FILE"] = previous_env_file

print("결과 코드:", outcome(closed_result), "/ state_saved:", closed_result.get("state_saved"))
print("메시지:", closed_result.get("message"))
print("사용자에게 한 질문 수:", len(closed_questions))
print("세션 이벤트 수:", closed_state["event_count"], "/ 기록된 답:", closed_state["answers"])
print("DAY04_ENV_FILE 복원됨:", os.environ.get("DAY04_ENV_FILE") == previous_env_file)

**관찰할 것.** 결과는 `decision_unavailable`, `state_saved: false`였고, 사용자에게 한 질문은 0개, 세션 이벤트는 시작과 제약 두 개 그대로입니다. 메시지에는 어떤 키 **이름**이 없었는지만 있고 값은 없습니다(`cap_lab/server/server.py:218-220`의 주석). 판정을 못 했으니 묻지도 기록하지도 않았습니다.

이 동작이 왜 중요한지 평가 쪽에서 보세요. H1의 목표 지표는 `vague_criteria` 사례의 `vague_not_frozen`(`cap_lab/scorers.py:196`)입니다. 게이트가 판정 실패 때 기준을 통과시킨다면, Decision API 장애가 난 날의 측정은 게이트를 끈 것과 같아지고 그 사실이 수치에 드러나지 않습니다. fail-closed는 장애를 `decision_unavailable`이라는 **보이는 오류**로 바꿉니다. 에이전트는 이것을 받고 다시 시도할지(코드 규칙: `retryable` 표시가 없으므로 다시 보내지 않음) 사용자에게 알릴지를 정합니다. 그리고 모델은 이 게이트를 우회할 인자를 갖고 있지 않습니다. 숨은 매개변수는 입력 형식에 없기 때문입니다.

### MRTR: 진짜 Jev로 한 번 수락, 한 번 거절

이제 판정기가 실제로 판정하게 합니다. 다음 두 셀은 각각 서버를 새로 띄우고 같은 모호한 기준 "분류가 잘 되면 완료"를 보냅니다. 서버 프로세스가 새로 뜨므로 판정 저장소도 비어 있고, 셀마다 Jev를 1회 부릅니다. 노트북 셀에는 Jev 호출이 보이지 않지만 **서버 프로세스가 `Day04_new/.env`의 키로 부르는 유료 호출**입니다.

`elicitation_callback(context, params)`이 이 노트북에서 **사용자 역할**을 합니다. `params.message`는 서버가 쓴 질문 문장이고, `params.requested_schema`는 받고 싶은 답의 형식(`RevisedCriteria`, `cap_lab/server/server.py:108-113`)입니다. 첫 셀은 시험 가능한 기준으로 수락하고, 둘째 셀은 거절합니다. 기록이 어떻게 남는지는 이벤트 payload의 `gate` 키로 읽습니다. 통과면 `source: "agent"`, 사용자에게 물었으면 `source: "elicited"`와 원래 문장(`original`)이 함께 남습니다(`cap_lab/server/server.py:283-289`, `cap_lab/server/workflow.py:205-209`).

Jev가 이 문장에 몇 점을 줄지는 실행해 봐야 압니다. 결과는 **1회 관찰**입니다. 점수가 0.5 이상으로 나와 질문 없이 기록되더라도 그것도 관찰 결과이고, 문턱값을 바꿀 근거는 아닙니다. 문턱값은 저자가 검토한 dev 라벨로만 조정합니다(사전 등록 P4).

**먼저 예측하세요.** Jev는 "분류가 잘 되면 완료"를 문턱값 0.5 아래로 볼까요? 아래로 본다면, 사용자가 수락한 이 한 번의 `record_answer`에 대해 서버의 호출 기록은 몇 개 남을까요?

In [ ]:
# [따라하기] (유료: Jev 1회, 서버가 부름) 모호한 완료 기준 -> 서버가 사용자에게 묻고 -> 시험 가능한 기준으로 수락
announce("MRTR 수락", model_calls=0, jev_calls=1)

VAGUE_CRITERION = "분류가 잘 되면 완료"
TESTABLE_CRITERION = "샘플 문의 50건 중 담당 분류와 일치하는 문의가 45건 이상이면 완료"
accept_dir = new_workdir("nb04-mrtr-accept")
accept_questions: list[dict] = []


async def answer_with_testable_criterion(context, params):
    """사용자 역할. 서버의 질문과 요청 형식을 적어 두고, 시험 가능한 완료 기준으로 수락한다."""
    accept_questions.append({
        "질문": params.message,
        "요청한 필드": sorted(params.requested_schema.get("properties", {})),
    })
    return ElicitResult(action="accept", content={"acceptance_criteria": TESTABLE_CRITERION})


async with connect_server(accept_dir, gate="h1", elicitation_callback=answer_with_testable_criterion) as client:
    accept_session = body(await client.call_tool("start_interview", {"goal": "고객 문의를 유형별로 나누는 도구"}))["session_id"]
    await client.call_tool("record_answer", {"session_id": accept_session, "field": "constraints", "answer": "없음"})
    accepted = body(await client.call_tool("record_answer", {
        "session_id": accept_session, "field": "acceptance_criteria", "answer": VAGUE_CRITERION,
    }))

print("결과:", outcome(accepted), "/ 상태:", accepted.get("status"))
print("기록된 완료 기준:", accepted.get("answers", {}).get("acceptance_criteria"))
print("서버가 사용자에게 한 질문 수:", len(accept_questions))
for question in accept_questions:
    print("   ", question)

# 이벤트 payload의 gate 키: score, threshold, source(agent 또는 elicited), original
accept_gates = [
    json.loads(path.read_text(encoding="utf-8"))["content"]["payload"].get("gate")
    for path in sorted((accept_dir / "evidence").glob("*/event-*.json"))
]
print("이벤트에 남은 게이트 판정:", [gate_record for gate_record in accept_gates if gate_record])
accept_calls = show_server_calls(accept_dir / "evidence")

**관찰할 것.** (1회 관찰) 결과는 셋 중 하나입니다.

- **점수가 0.5 미만이었다면**: 질문이 1개 왔고, 기록된 완료 기준은 `TESTABLE_CRITERION`이며, 게이트 기록은 `source: "elicited"`와 `original: "분류가 잘 되면 완료"`를 담습니다. 호출 기록 표에는 `record_answer`가 완료 기준에 대해 **두 줄** 있습니다. 첫 줄은 `inputRequests`(서버가 질문을 돌려준 중간 결과), 둘째 줄은 `success`(답과 함께 다시 보낸 같은 요청)입니다. MRTR이 같은 `tools/call`을 두 번 보내는 구조이고, 2절에서 본 멱등성이 여기서 필요해집니다. 사용자가 준 문장은 다시 판정하지 않습니다. 사람이 직접 준 기준을 우선합니다.
- **점수가 0.5 이상이었다면**: 질문 없이 원래 문장이 기록되고 게이트 기록은 `source: "agent"`입니다. Jev가 이 문장을 판정 가능하다고 본 것입니다. 이것은 1회 관찰이고 문턱값을 바꿀 근거가 아닙니다.
- **`decision_unavailable`이었다면**: Jev 호출이 실패한 것입니다. 키 설정(00 노트북의 키 표)과 네트워크를 확인하세요. 아무것도 기록되지 않았습니다.

어느 경우든 모델은 끼어들 수 없었습니다. 대체 기준은 `elicitation_callback`(사용자)만 줄 수 있고, `record_answer`의 입력 형식에는 그것을 넣을 칸이 없습니다.

In [ ]:
# [따라하기] (유료: Jev 1회, 서버가 부름) 같은 모호한 기준 -> 사용자가 거절
announce("MRTR 거절", model_calls=0, jev_calls=1)

decline_dir = new_workdir("nb04-mrtr-decline")
decline_questions: list[str] = []


async def decline_the_question(context, params):
    """사용자 역할. 질문을 적어 두고 대체 기준을 주지 않고 거절한다."""
    decline_questions.append(params.message)
    return ElicitResult(action="decline")


async with connect_server(decline_dir, gate="h1", elicitation_callback=decline_the_question) as client:
    decline_session = body(await client.call_tool("start_interview", {"goal": "고객 문의를 유형별로 나누는 도구"}))["session_id"]
    await client.call_tool("record_answer", {"session_id": decline_session, "field": "constraints", "answer": "없음"})
    declined = body(await client.call_tool("record_answer", {
        "session_id": decline_session, "field": "acceptance_criteria", "answer": VAGUE_CRITERION,
    }))
    freeze_after_decline = body(await client.call_tool("freeze_seed", {"session_id": decline_session}))
    decline_state = body(await client.call_tool("get_session", {"session_id": decline_session}))

print("결과:", outcome(declined), "/", declined.get("message", "(메시지 없음: 기록됨)"))
print("서버가 사용자에게 한 질문 수:", len(decline_questions))
print("거절 뒤 확정 시도:", outcome(freeze_after_decline), "/ 남은 항목:", freeze_after_decline.get("missing_fields"))
print("세션 상태:", decline_state["status"], "/ 이벤트 수:", decline_state["event_count"])
decline_calls = show_server_calls(decline_dir / "evidence")

**관찰할 것.** (1회 관찰) 점수가 문턱값 미만이었다면 결과는 `criteria_not_testable`이고 아무것도 기록되지 않았습니다. 이어서 시도한 `freeze_seed`는 `seed_incomplete`로 거절됩니다. 완료 기준이 비어 있으면 확정할 수 없기 때문입니다(`cap_lab/server/workflow.py:222-227`). 즉 모호한 기준은 **구조적으로** 확정 명세가 될 수 없습니다. dev-06 사례가 이 경로를 잽니다. 그 사례는 대본 응답(`elicitation`)이 비어 있어 실행기가 거절하고(`cap_lab/experiment.py:303-308`), 기대값이 `frozen: false`입니다.

`decision_unavailable`과 `criteria_not_testable`을 구분하세요. 앞의 것은 **판정을 못 했다**(장애)이고, 뒤의 것은 **판정했고, 물었고, 사용자가 시험 가능한 기준을 주지 않았다**(업무 결과)입니다. 둘 다 아무것도 기록하지 않지만, 에이전트가 사용자에게 할 말은 다릅니다. 점수가 0.5 이상으로 나와 질문이 오지 않았다면 이 셀의 기준은 기록되고 확정까지 갔을 것입니다. 그 경우도 그대로 1회 관찰로 적어 두세요.

운영 관점에서 한 가지를 더 기억해 두세요. MRTR의 중간 상태(`request_state`)는 기본 설정에서 **서버 프로세스마다 다른 키**로 서명되므로, 사용자가 답하는 사이에 서버가 재시작되면 그 상태는 거절됩니다(`.venv/lib/python3.13/site-packages/mcp/client/client.py:772-776`). 1절의 세션은 DB와 파일 덕분에 재시작을 견디지만, 진행 중인 질문 하나는 견디지 못합니다. 이 노트북은 그 경우를 실행해 보지 않았습니다.

## 8. 인증은 개념만: DCR에서 Client ID Metadata Documents로

4절의 HTTP 서버는 인증이 없어서 `127.0.0.1`에만 열었습니다. 이 컴퓨터 밖으로 MCP 서버를 열려면 인증이 필요합니다. MCP의 HTTP 인증은 OAuth 2.1을 따릅니다. MCP 서버는 보호된 자원(protected resource)이고, 클라이언트는 인가 서버(authorization server)에서 받은 접근 토큰을 요청에 붙입니다.

어려운 부분은 토큰보다 앞에 있습니다. 인가 서버는 토큰을 주기 전에 **이 클라이언트가 누구인지**(`client_id`) 알아야 합니다. 사람이 미리 등록해 두는 방식은 MCP처럼 처음 보는 클라이언트가 수시로 붙는 생태계에 맞지 않습니다. 그래서 그동안은 **DCR**(OAuth 2.0 Dynamic Client Registration)을 썼습니다. 클라이언트가 실행 중에 인가 서버의 등록 엔드포인트에 자기 정보를 보내고 `client_id`를 받아 가는 방식입니다. 문제는 세 가지였습니다. 등록 엔드포인트를 누구에게나 열어 두어야 하고, 클라이언트가 붙을 때마다 일회용 등록 기록이 쌓이고, 인가 서버는 등록하러 온 클라이언트가 자칭하는 그 클라이언트인지 확인할 방법이 없습니다.

**Client ID Metadata Documents(CIMD)** 는 등록 단계를 없앱니다. `client_id`가 HTTPS URL이고, 그 URL에는 클라이언트의 메타데이터(이름, redirect URI 등)를 담은 JSON 문서가 있습니다. 인가 서버는 등록 요청을 받는 대신 그 URL에서 문서를 가져와 확인합니다. 신뢰는 "그 도메인을 통제하는 쪽이 그 클라이언트다"에 묶입니다. 과정 설계 문서도 "DCR은 Client ID Metadata Documents로 대체 예정"이라고 적어 두었습니다(`docs/DESIGN.md:61`).

설치된 mcp 2.3.0 클라이언트에서 확인할 수 있는 것은 여기까지입니다.

- 인가 서버 메타데이터에 `client_id_metadata_document_supported` 필드가 있습니다(`.venv/lib/python3.13/site-packages/mcp/shared/auth.py:229`).
- `OAuthClientProvider`는 `client_metadata_url`을 받습니다. 인가 서버가 지원한다고 알리면 이 URL을 `client_id`로 쓰고 DCR을 하지 않으며, URL은 루트가 아닌 경로를 가진 HTTPS여야 합니다(`.venv/lib/python3.13/site-packages/mcp/client/auth/oauth2.py:307`, `318-321`).
- 둘 중 무엇을 쓸지는 `should_use_client_metadata_url`이 정하고, 조건이 맞지 않으면 DCR로 돌아갑니다(`.venv/lib/python3.13/site-packages/mcp/client/auth/utils.py:372-395`).

이 과정에는 인가 서버가 없으므로 인증 코드는 실행하지 않습니다. 인증을 붙인다면 4절 서버의 주소 고정(`127.0.0.1`)을 푸는 것은 **인증을 붙인 뒤**여야 한다는 순서만 기억하세요.

## 9. 에이전트로 한 번: dev-05를 H1 게이트로 실행

마지막으로 Deep Agent가 H1 게이트를 만나는 장면을 한 번 봅니다. 사례는 dev-05입니다. 고객은 "고객 문의를 유형별로 나눠 주는 도구"를 원하고, 제약은 없음, 완료 기준은 "분류가 잘 되면 완료"라고 말한 뒤 **바로 명세를 확정해 달라**고 합니다. 사례 파일에는 서버가 물으면 줄 대본 응답도 적혀 있습니다(시험 가능한 기준 하나). 사용자 역할은 LLM이 흉내 내지 않고, 실행기가 이 대본으로만 답합니다(`cap_lab/experiment.py:303-308`, 사전 등록 P4). 대본에 요청 필드가 모두 있으면 수락하고 하나라도 없으면 거절합니다.

`run_case`(`cap_lab/experiment.py:129`)는 새 DB와 새 근거 폴더에서 서버를 `--gate h1`로 띄우고 Deep Agent를 만들어 사례의 발화를 보냅니다. 서버의 MRTR 질문은 에이전트 쪽에서 `mcp_elicitation` 종류의 LangGraph interrupt로 올라오고, 실행기는 대본 응답으로 재개합니다(`cap_lab/experiment.py:226-234`). 실행이 끝나면 `score_run`(`cap_lab/scorers.py:299`)이 결정적 검사를 계산하고, `first_failure`(`cap_lab/scorers.py:322`)가 가장 먼저 실패한 단계를 알려 줍니다.

규모는 Claude 약 9회, Jev 최대 3회입니다. 사례당 Claude 호출 추정 5~9회(`cap_lab/experiment.py:50`)와 게이트 하나가 실행 한 번에 Jev를 부르는 상한 추정 3회(`cap_lab/experiment.py:52`)에서 왔습니다. 반복 없이 한 번이므로 결과는 **1회 관찰**입니다. 개선 여부는 07에서 A/A와 쌍체 비교로 잽니다.

**먼저 예측하세요.** 에이전트는 모호한 기준을 `record_answer`로 보낼까요, 아니면 서버에 보내기 전에 사용자에게 글로 되물을까요? 명세가 확정된다면 어느 완료 기준으로 확정될까요? `vague_not_frozen`과 `frozen_ok`는 각각 무엇이 나올까요?

In [ ]:
# [따라하기] (유료) dev-05를 H1 게이트를 켠 Deep Agent로 한 번 실행한다
announce("dev-05 H1 에이전트 1회", model_calls=9, jev_calls=3, runs=1)

from cap_lab.experiment import Variant, load_cases, run_case
from cap_lab.scorers import first_failure, score_run
from cap_lab.settings import load_settings

# 1) dev 사례 파일에서 dev-05만 꺼낸다.
dev05_case = next(case for case in load_cases(split="dev") if case["id"] == "dev-05")
print("사용자 발화:", dev05_case["turns"][0])
print("대본 응답:", dev05_case["elicitation"])

# 2) H1 게이트만 켠 구성으로 한 번 실행한다. 실패는 예외가 아니라 record.error에 남는다.
h1_variant = Variant(name="h1", gate_h1=True)
agent_dir = new_workdir("nb04-agent-h1")
agent_record = await run_case(dev05_case, variant=h1_variant, settings=load_settings(), workdir=agent_dir, repeat=1)

print("\n실행 오류:", agent_record.error or "없음")
print("모델 호출 수:", agent_record.model_calls, "/ 실행 시간(초):", round(agent_record.latency_s, 1))

# 3) 에이전트가 부른 Tool 순서와 인자.
show_table([
    {"순서": index, "Tool": call["name"], "성공": call["ok"],
     "인자": json.dumps(call["args"], ensure_ascii=False)[:120]}
    for index, call in enumerate(agent_record.tool_calls, start=1)
])

# 4) 서버가 한 질문과 실행기가 준 대본 응답, 그리고 H1 판정 기록.
print("사용자에게 온 질문과 대본 응답:")
for item in agent_record.interrupts:
    print("   ", item)
print("H1 판정 기록:", agent_record.gate_decisions)

# 5) 결정적 검사. None은 '해당 없음 또는 확인 불가'이고 실패가 아니다.
agent_scores = score_run(agent_record)
show_table([{"검사": score.name, "값": score.value, "설명": score.comment} for score in agent_scores])
print("가장 먼저 실패한 단계:", first_failure(agent_scores))
show_events(agent_record.evidence_dir)

**관찰할 것.** (1회 관찰) 세 곳을 이어서 읽으세요.

- **Tool 순서**: 에이전트가 `record_answer`로 "분류가 잘 되면 완료"를 보냈다면, 질문·대본 응답 목록에 서버의 질문이 하나 있고 응답은 `accept`였을 것입니다. 이벤트 표의 완료 기준은 대본의 시험 가능한 문장이고, 게이트 기록은 `source: "elicited"`입니다. 모델은 이 대체 문장을 만들지 않았습니다.
- **검사**: dev-05의 기대는 `must_call: start_interview`, 모호한 답이 확정되지 않을 것(`vague_not_frozen`), 확정될 것(`frozen_ok`)입니다. 에이전트가 서버에 보내기 전에 사용자에게 글로 되묻고 멈췄다면 MRTR은 열리지 않고 `frozen_ok`가 실패할 수 있습니다. 게이트가 막은 것이 아니라 에이전트가 먼저 멈춘 것입니다. 점수만 보지 말고 Tool 순서로 이유를 확인하세요.
- **비용**: 실제 모델 호출 수를 고지한 9회와 비교해 보세요. 크게 넘었다면 어디서 반복했는지 Tool 순서에서 찾을 수 있습니다.

이 한 번의 결과로 "H1이 S3를 개선한다"고 말할 수는 없습니다. 같은 사례를 게이트 없이 돌린 결과와, 잡음 폭을 재는 A/A 비교가 함께 있어야 합니다. 그것이 07의 일입니다.

## 직접 해보기

세 과제는 모두 위에서 정의한 도우미(`body`, `outcome`, `show_server_calls`, `call_with_retry`)를 그대로 씁니다. 해보기 1, 2, 4는 무료이고, 해보기 3만 Jev를 부르므로 `RUN_EXERCISE = False`로 잠겨 있습니다.

### 해보기 1. 내 장애 계획과 재시도 상한

`freeze_seed`가 처음 두 번 일시 장애를 내는 서버에서 인터뷰를 확정까지 끌고 가 보세요. 아래 셀의 `MY_FAULT`와 `MY_MAX_ATTEMPTS` 두 줄만 바꾸면 됩니다. 먼저 예측해 보세요. `MY_MAX_ATTEMPTS = 2`로 줄이면 확정이 될까요? 그때 근거 폴더의 `seed_frozen` 이벤트는 몇 개일까요? `retry_bounded` 검사(`cap_lab/scorers.py:252`)의 상한 `실패 횟수 + 2`와 비교해 내 상한은 안전한가요? 마지막으로 `transient` 대신 `permission_denied: ["freeze_seed"]`를 넣으면 `call_with_retry`가 몇 번 보내는지도 확인하세요.

In [ ]:
# [내 것 만들기] 장애 계획과 재시도 상한을 정하고, 명세 확정까지 가 본다 (무료)
MY_FAULT = FaultPlan(transient={"freeze_seed": 2})  # 바꿔 볼 곳 1: 어떤 Tool을 몇 번 실패시킬지
MY_MAX_ATTEMPTS = 3                                  # 바꿔 볼 곳 2: 재시도 상한

exercise_retry_dir = new_workdir("nb04-exercise-retry")
async with connect_server(exercise_retry_dir, fault=MY_FAULT) as client:
    my_session = body(await client.call_tool("start_interview", {"goal": GOAL}))["session_id"]
    my_answers = [
        ("constraints", CONSTRAINT),
        ("acceptance_criteria", "샘플 영수증 20장을 넣으면 합계가 수기 정산표와 1원 단위까지 일치한다"),
    ]
    for field, answer in my_answers:
        content, sent = await call_with_retry(
            client, "record_answer", {"session_id": my_session, "field": field, "answer": answer},
            max_attempts=MY_MAX_ATTEMPTS,
        )
        print(f"record_answer({field}): {outcome(content)} / 보낸 횟수 {sent}")
    my_freeze, sent = await call_with_retry(client, "freeze_seed", {"session_id": my_session}, max_attempts=MY_MAX_ATTEMPTS)
    print(f"freeze_seed: {outcome(my_freeze)} / 보낸 횟수 {sent} / 상태 {my_freeze.get('status')}")

exercise_calls = show_server_calls(exercise_retry_dir / "evidence")
show_events(exercise_retry_dir / "evidence")

### 해보기 2. 근거 폴더만 넘겨받아 이어 가기

다른 담당자가 1절의 세션을 넘겨받았다고 해 봅시다. 받은 것은 근거 폴더뿐입니다. 아래 셀은 근거 폴더를 새 작업 폴더로 복사하고, 빈 DB로 서버를 띄워 목록에서 세션을 찾은 뒤, 남은 질문에 답하고 확정하고, 확정 명세를 원본 근거로 연결한 산출물을 적재합니다. `HANDOFF_FROM`을 `idem_dir`(2절에서 이미 확정한 세션)로 바꾸면 어떤 단계가 건너뛰어지는지, `fault_dir`(3절)로 바꾸면 무엇이 달라지는지 예측하고 실행해 보세요. 산출물의 `source_evidence_id`가 확정 명세 ID를 가리키는 것이 06 노트북의 `evidence_linked` 검사가 보는 연결입니다.

In [ ]:
# [바꿔보기] 근거 폴더만 넘겨받아 남은 질문에 답하고, 확정하고, 근거를 연결한 산출물을 적재한다 (무료)
HANDOFF_FROM = persist_dir  # 바꿔 볼 곳: idem_dir 또는 fault_dir로 바꿔 보세요.

handoff_dir = new_workdir("nb04-exercise-handoff")
shutil.copytree(HANDOFF_FROM / "evidence", handoff_dir / "evidence")

async with connect_server(handoff_dir) as client:
    # 1) DB가 없으므로 근거 폴더의 세션 목록에서 이어 갈 세션을 고른다.
    handoff_sessions = body(await client.call_tool("list_evidence", {}))["sessions"]
    print("넘겨받은 세션:", [(item["session_id"][:8], item["goal"]) for item in handoff_sessions])
    handoff_session = handoff_sessions[0]["session_id"]
    handoff_state = body(await client.call_tool("get_session", {"session_id": handoff_session}))
    print("복원한 상태:", handoff_state["status"], "/ 남은 항목:", handoff_state["missing_fields"])

    # 2) 남은 항목이 있으면 답한다.
    planned_answers = {"constraints": "없음", "acceptance_criteria": "샘플 영수증 20장의 합계가 수기 정산표와 1원 단위까지 일치한다"}
    for field in handoff_state["missing_fields"]:
        answered = body(await client.call_tool(
            "record_answer", {"session_id": handoff_session, "field": field, "answer": planned_answers[field]}
        ))
        print(f"record_answer({field}):", outcome(answered))

    # 3) 확정한다. 이미 확정된 세션이면 같은 명세를 돌려준다.
    handoff_seed = body(await client.call_tool("freeze_seed", {"session_id": handoff_session}))
    print("확정:", outcome(handoff_seed), "/ 확정 명세 근거 ID:", handoff_seed.get("seed_evidence_id"))

    # 4) 확정 명세를 원본 근거로 연결한 산출물을 적재한다.
    if handoff_seed.get("success"):
        artifact = body(await client.call_tool("save_evidence", {
            "session_id": handoff_session, "title": "인수인계 메모",
            "content": f"확정 명세 {handoff_seed['seed_evidence_id']}를 근거로 구현을 시작한다.",
            "source_evidence_id": handoff_seed["seed_evidence_id"],
        }))
        print("산출물:", outcome(artifact), "/ 원본 근거:", artifact.get("evidence", {}).get("source_evidence_id"))

show_events(handoff_dir / "evidence")

### 해보기 3. 문턱값을 바꿔 MRTR이 열리는 조건 보기 (유료, 잠김)

같은 완료 기준을 문턱값 두 개(0.1, 0.9)로 판정하게 하고, 서버가 사용자에게 묻는지 봅니다. 문턱값마다 서버를 새로 띄우므로 Jev는 문턱값 하나에 1회, 모두 최대 2회입니다. 이 과제는 **장치의 동작을 보는 것**이지 문턱값을 고르는 것이 아닙니다. 측정에 쓰는 문턱값은 저자가 검토한 dev 라벨로만 조정합니다(사전 등록 P4). `MY_CRITERION`을 시험 가능한 문장과 모호한 문장으로 바꿔 가며, 같은 점수가 문턱값에 따라 어떻게 다른 행동이 되는지 비교하세요. 실행하려면 `RUN_EXERCISE = True`로 바꿉니다(본인의 Decision API 크레딧을 씁니다).

In [ ]:
# [바꿔보기] 문턱값 두 개로 같은 완료 기준을 판정하게 한다 (켜면 Jev 최대 2회)
RUN_EXERCISE = False  # True로 바꾸면 서버가 Jev를 최대 2회 부릅니다.
MY_CRITERION = "보기 좋게 요약되면 끝"  # 바꿔 볼 곳: 시험 가능한 문장으로도 바꿔 보세요.
THRESHOLDS = [0.1, 0.9]

if RUN_EXERCISE:
    announce("문턱값 비교", model_calls=0, jev_calls=2)
    threshold_rows = []
    for threshold in THRESHOLDS:
        asked_messages: list[str] = []

        async def note_and_decline(context, params, asked_messages=asked_messages):
            """질문이 왔는지만 적고 거절한다."""
            asked_messages.append(params.message)
            return ElicitResult(action="decline")

        threshold_dir = new_workdir(f"nb04-exercise-threshold-{threshold}")
        async with connect_server(threshold_dir, gate="h1", threshold=threshold,
                                  elicitation_callback=note_and_decline) as client:
            threshold_session = body(await client.call_tool("start_interview", {"goal": "주간 매출 요약 보고 도구"}))["session_id"]
            await client.call_tool("record_answer", {"session_id": threshold_session, "field": "constraints", "answer": "없음"})
            judged = body(await client.call_tool("record_answer", {
                "session_id": threshold_session, "field": "acceptance_criteria", "answer": MY_CRITERION,
            }))
        gates = [json.loads(path.read_text(encoding="utf-8"))["content"]["payload"].get("gate")
                 for path in (threshold_dir / "evidence").glob("*/event-*.json")]
        threshold_rows.append({
            "문턱값": threshold, "결과": outcome(judged), "질문 수": len(asked_messages),
            "게이트 기록": [gate_record for gate_record in gates if gate_record],
        })
    show_table(threshold_rows)
else:
    print("RUN_EXERCISE = False 라서 실행하지 않았습니다. 켜면 서버가 Jev를 최대 2회 부릅니다.")

### 해보기 4. 추적 하나로 여러 Tool 호출 묶기

6절의 깨뜨리기에서는 부모 span이 없어 요청마다 trace-id가 달랐습니다. 이번에는 인터뷰 한 번(시작, 제약, 완료 기준, 확정)을 **하나의 부모 span**으로 감싸고, 서버가 받은 모든 요청의 trace-id가 하나인지 확인하세요. 그다음 부모 span 안에 단계별 자식 span(`"질문 1"`, `"확정"`)을 열어 보고, 서버가 받은 traceparent의 parent-id가 어느 span을 가리키는지 `span_exporter`의 span 목록과 맞춰 보세요. 에이전트 실행에서 Langfuse observation이 하는 일이 바로 이 부모 span 역할입니다.

In [ ]:
# [내 것 만들기] 인터뷰 한 번을 하나의 trace로 묶는다 (무료)
meta_log.unlink(missing_ok=True)
span_exporter.clear()

with tracer.start_as_current_span("인터뷰 한 번") as interview_span:  # 바꿔 볼 곳: 단계별 자식 span을 더해 보세요.
    async with Client(observer_params) as client:
        my_traced_session = body(await client.call_tool("start_interview", {"goal": GOAL}))["session_id"]
        await client.call_tool("record_answer", {"session_id": my_traced_session, "field": "constraints", "answer": "없음"})
        await client.call_tool("record_answer", {
            "session_id": my_traced_session, "field": "acceptance_criteria",
            "answer": "샘플 영수증 20장의 합계가 수기 정산표와 1원 단위까지 일치한다",
        })
        await client.call_tool("freeze_seed", {"session_id": my_traced_session})

interview_trace_id = format(interview_span.get_span_context().trace_id, "032x")
my_received = [json.loads(line) for line in meta_log.read_text(encoding="utf-8").splitlines()]
my_trace_ids = {parse_traceparent(item["meta"].get("traceparent", ""))["trace-id"] for item in my_received}
print("서버가 받은 요청 수:", len(my_received))
print("서로 다른 trace-id 수:", len(my_trace_ids), "/ 부모 span과 같은가:", my_trace_ids == {interview_trace_id})
show_table([{"span 이름": span.name, "span id": format(span.context.span_id, "016x"),
             "부모 span id": format(span.parent.span_id, "016x") if span.parent else ""}
            for span in span_exporter.get_finished_spans()])

## 확인 질문

1. 1절에서 서버 프로세스를 바꿔도 세션이 이어진 이유를 `replay`와 `_load`라는 두 이름으로 설명해 보세요. 서버가 상태를 메모리에 캐시해 두었다면 무엇이 깨졌을까요?
2. 근거 파일의 답변 문장을 바꿨을 때 서버가 DB와 파일 중 어느 쪽도 고르지 않고 `evidence_conflict`로 멈춘 것은 장점인가요, 단점인가요? 운영자가 그 세션을 살리려면 무엇을 확인해야 할까요?
3. `record_answer`는 `idempotentHint: true`이고 `start_interview`는 `false`입니다. 이 힌트를 실제로 지키게 만드는 코드는 어디에 있고, 힌트만 있고 코드가 없다면 R3 검사는 무엇을 보게 될까요?
4. 재시도 라우팅은 코드 규칙이고 완료 기준의 판정 가능성은 Jev 판단입니다. 두 결정을 가르는 기준을 사전 등록 P3의 병목 조건(판단 형태인가, 결정적 규칙이 없는가)으로 설명해 보세요.
5. stdio와 Streamable HTTP에서 서버 코드는 한 줄도 바뀌지 않았습니다. 무엇이 그것을 가능하게 했고, HTTP로 바꾸면서 새로 생긴 운영 책임은 무엇인가요?
6. 이 서버의 `tools/list`에 `ttl_ms=600000`, `cache_scope="public"`을 준다면 어떤 상황에서 문제가 생길까요?
7. 서버가 받은 traceparent와 근거 폴더의 호출 기록이 지금은 이어지지 않습니다. 왜 그런지, 그리고 잇는다면 어떤 값을 어디에 남겨야 하는지 말해 보세요.
8. fail-closed에서 `decision_unavailable`과 MRTR 거절의 `criteria_not_testable`은 둘 다 아무것도 기록하지 않습니다. 에이전트가 사용자에게 해야 할 말은 각각 어떻게 다른가요?

## 정리

- **영속화와 재생**: 이 서버의 상태는 SQLite 이벤트와 근거(Evidence) 파일에만 있고, 현재 상태는 요청마다 `replay`로 다시 계산합니다. 그래서 서버 프로세스를 바꿔도, DB를 잃어도 근거 파일로 이어 갑니다. 기록이 서로 어긋나면 고르지 않고 `evidence_conflict`·`evidence_corrupt`로 멈춥니다.
- **멱등성**: `record_answer`, `freeze_seed`, `save_evidence`는 같은 요청을 다시 받아도 상태가 한 번만 바뀌고, `start_interview`는 그렇지 않습니다. ToolAnnotations는 힌트이고, 실제 보장은 `workflow.py`의 코드입니다. R3 검사는 호출 수가 아니라 상태 이벤트 수를 셉니다.
- **장애와 재시도**: `--fault`의 권한 거부와 일시 장애는 업무 코드 앞에서 부작용 없이 업무 오류로 돌아옵니다. 재시도는 오류 코드와 `retryable`로 정하는 코드 규칙입니다. Jev의 자리가 아닙니다.
- **Streamable HTTP**: 같은 서버를 `127.0.0.1`의 독립 프로세스로 띄우면 여러 클라이언트가 같은 세션을 봅니다. 대신 서버를 띄우고 끄는 책임과 인증 책임이 생깁니다.
- **ttlMs와 cacheScope**: mcp 2.3.0은 목록·읽기 결과에 두 힌트를 싣고 클라이언트가 그것을 따릅니다. 이 서버는 힌트를 주지 않아 SDK 기본값(`0`, `private`)이 나가고, 클라이언트는 매번 서버까지 갑니다.
- **traceparent**: mcp 클라이언트는 현재 OpenTelemetry 문맥을 요청 `_meta`에 실어 보냅니다. 여러 호출을 한 trace로 묶는 것은 부모 span입니다. 이 서버의 호출 기록은 `_meta`를 남기지 않고 서버 span은 Langfuse로 내보내지 않으므로, 서버 쪽 근거와 Langfuse trace는 아직 이어지지 않습니다.
- **H1 게이트**: 완료 기준 판정은 서버가 Jev로 하고, 대체 기준은 MRTR로 사용자에게서만 받습니다. 모델은 그 인자를 볼 수도 채울 수도 없습니다. 판정기를 쓸 수 없으면 `decision_unavailable`로 닫힙니다.
- **인증**: HTTP를 이 컴퓨터 밖으로 열려면 OAuth 기반 인증이 필요하고, 클라이언트 식별은 DCR에서 Client ID Metadata Documents로 옮겨 가고 있습니다. 이 과정에서는 개념만 다뤘습니다.

이 노트북에서 직접 확인하지 않은 것도 적어 둡니다. 서버 span의 Langfuse 내보내기, 서버 재시작을 사이에 둔 MRTR 재개, 인증 흐름, 그리고 Jev가 이 노트북의 문장에 주는 점수(실행할 때마다 1회 관찰)입니다. 다음 05는 이 MCP Tool들을 Skill을 읽은 뒤에만 모델에 보이게 하는 구성을 다룹니다.